# CKG → Knowledge Graph (KG) Builder (Stage 2)

**Source:** CKG (Clinical Knowledge Graph) TSV files  
**Role:** Full ID standardization and KG schema harmonization across all CKG relation types

## What this notebook does

Loads CKG TSV files and applies full annotation and ID normalisation:
- **Drug nodes:** DrugBank ID → PubChem CID; IUPAC name and SMILES annotated
- **Gene nodes:** synonym expansion via NCBI → canonical symbol → Ensembl ID
- **Protein nodes:** UniProt ID → RecName annotation
- **Disease nodes:** DOID name + xrefs; ICD-prefixed IDs filtered out
- **Metabolite nodes:** HMDB ID → common name + IUPAC (via HMDB dict)
- **Tissue nodes:** BTO ID → tissue name; non-BTO Tails filtered out
- **GO/Ontology nodes:** GO ID → term name + namespace

## Files processed and outputs

| File # | Source TSV | Head type | Tail type | Output CSV |
|--------|-----------|-----------|-----------|------------|
| 1 | dgidb_targets | Drug | Gene | File_1_Drug_Gene_CKG.csv |
| 2 | cgi_targets_clinically_relevant_variant | Drug | Mutation_Variant | File_2_Drug_MutationVariant_CKG.csv |
| 3 | oncokb_targets_clinically_relevant_variant | Drug | Mutation_Variant | File_3_Drug_MutationVariant_OncoKB_CKG.csv |
| 4 | oncokb_targets | Drug | Gene | File_4_Drug_Gene_OncoKB_CKG.csv |
| 6 | protein_is_biomarker_of_disease | Protein_BioM | Disease | File_6_ProtBioM_Disease_CKG.csv |
| 7 | protein_is_qcmarker_in_tissue | Protein_QC_BioM | Tissue_Ontology | File_7_ProtQC_bioM_Tissue_CKG.csv |
| 8-10 | GWAS files | — | — | **DROPPED** (not relevant) |
| 11 | Gene_known_variant_found_in_gene | Mutation_Variant | Gene | File_11_Mutation_Gene.csv |
| 12 | Biological_process_associated_with | Protein | Ontology | File_12_Protein_BiologicalProcess_CKG.csv |
| 13 | Cellular_component_Publication | PMID | Cellular_component | File_13_PMID_Cellular_component_CKG.csv |
| 14 | Cellular_component_associated_with | Protein | Cellular_component | File_14_Protein_Cellular_component_CKG.csv |
| 15 | Chromosome file | — | — | **DROPPED** (Chromosome not in scope) |
| 16 | Disease_Publication | PMID | Disease | File_16_PMID_Disease_CKG.csv |
| 17 | Metabolite_Publication | Metabolite | PMID | File_17_Metabolite_pubmed_CKG.csv |
| 18 | Molecular_function_associated_with | Protein | Ontology | File_18_protein_MolecularFunction_CKG.csv |
| 19 | Peptide_belongs_to_protein | — | — | **DROPPED** (peptide sequences not suitable) |
| 20 | Protein_Cellular_component_integrated | Protein | Cellular_component | File_20_protein_cellular_compartment_CKG.csv |
| 21 | Protein_Disease_associated_with_integrated | Protein | Disease | File_21_protein_Disease_CKG.csv |
| 23 | Protein_Tissue_associated_with_integrated | Protein | Tissue | File_23_protein_tissue_CKG.csv |
| 24 | Protein_belongs_to_taxonomy | Protein | Taxanomy | (inspect only) |
| 25 | Protein_disgenet_associated_with | Protein | Disease | File_25_protein_disease_CKG.csv |
| 26 | Protein_gene_translated_into | Gene | Protein | File_26_Gene_protein_CKG.csv |
| 27 | Protein_has_structure | Protein | PDB | File_27_Protein_PDB_CKG.csv |
| 28 | Protein_known_variant_found_in_protein | Mutation | Protein | File_28_Mutation_Protein_CKG.csv |
| 28_1 | Protein_transcript_translated_into | Transcript | Protein | File_28_1_Transcript_Protein_CKG.csv |
| 29 | Transcript_is_isoform | Protein | Protein | File_29_Protein_Protein_CKG.csv |
| 30 | cgi_targets | Drug | Gene | File_30_Drug_Gene_CKG.csv |
| 31 | hmdb_associated_with_disease | Metabolite | Disease | File_31_Metabolite_Disease_CKG.csv |
| 32 | hmdb_associated_with_protein | Metabolite | Protein | File_32_Metabolite_Protein_CKG.csv |
| 33 | intact_interacts_with | Protein | Protein | File_33_Protein_Protein_CKG.csv |
| 34 | mutation_curated_affects_interaction_with | Mutation | Protein | File_34_Mutation_Protein_CKG.csv |
| 36 | refseq_transcribed_into | Gene | Ref_Sequence | File_36_Gene_Transcript_CKG.csv |
| 37 | sider_is_indicated_for | Drug | Phenotype | File_37_Drug_Phenotype_sideffect_CKG.csv |
| 38 | stitch_associated_with | Drug | Protein | File_38_Drug_Protein_CKG.csv |
| 39 | stitch_drug_acts_on_protein | Drug | Protein | File_39_Drug_Protein_CKG.csv |
| 40 | string_interacts_with | Protein | Protein | File_40_Protein_Protein_CKG.csv |
| 41 | string_protein_acts_on_protein | Protein | Protein | File_41_Protein_Protein_CKG.csv |
| 42 | Tissue_Publication | PMID | Tissue | File_42_PMID_Tissue_CKG.csv |

## Bugs fixed from original


---
## 0 · Configuration — edit ONLY these two lines

In [ ]:
/ckg/data/imports/
ckg/docker_data/neo4j/exports/nodes/relations/curated/protein_is_biomarker_of_disease.tsv'
df6 = process_dataframe(
    os.path.join(CKG_DOC_PATH, 'curated/protein_is_biomarker_of_disease.tsv'),
    head_type='Protein_BioM', tail_type='Disease')

In [2]:
import os
import numpy as np
import pandas as pd
from glob import glob

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/"

# ── Derived input paths (do not edit below this line) ────────────────────────
PUBCHEM_PKL_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.paraquet")
PUBCHEM_DBCHEBI_PATH = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv")
DB_SMILES_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/drugbank/DB_2_SMILES.txt")
DB_ALL_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/drugbank/drugbank_all.csv")
ENS2NCBI_PATH      = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
NCBI_HUMAN_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
DO_PATH            = os.path.join(BASE_PATH, "databases_for_mapping/DO/All_DO_ref_files.csv")
UNIPROT_ALL_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/uniprot/UNIPROT_NCBI_ALL_MAPPED.csv")
UNIPROT_REC_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/uniprot/Uniprot_ID_Recname.csv")
BTO_PATH           = os.path.join(BASE_PATH, "databases_for_mapping/bto/BTO_ALL_COMBINED.csv")
GO_PATH            = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_GO_ID_NAME.csv")
HMDB_PATH          = os.path.join(BASE_PATH, "databases_for_mapping/hmdb/HMDBoutput.csv")
PHENOTYPE_PATH     = os.path.join(BASE_PATH, "ckg/docker_data/neo4j/exports/nodes/Phenotype.tsv")


# CKG database TSV files — first set (non-Docker)
CKG_DB_PATH  = os.path.join(BASE_PATH, "ckg/data/imports/databases/")
# CKG Docker database TSV files
CKG_DOC_PATH = os.path.join(BASE_PATH, "/ckg/data/imports/")

os.makedirs(OUT_PATH, exist_ok=True)

print("Paths configured successfully.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output dir  : {OUT_PATH}")

Paths configured successfully.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output dir  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/


---
## 1 · Load all reference dictionaries

In [5]:
# ── 1a. PubChem CID → IUPAC name and SMILES ──────────────────────────────────
Pubchem_full = pd.read_parquet(PUBCHEM_PKL_PATH)
Pubchem_Smile_CID_Dict = dict(zip(Pubchem_full['PUBCHEM_COMPOUND_CID'], Pubchem_full['PUBCHEM_SMILES']))
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem_full['PUBCHEM_COMPOUND_CID'], Pubchem_full['PUBCHEM_IUPAC_NAME']))
del Pubchem_full
print(f"PubChem CID dicts loaded")

PubChem CID dicts loaded


In [6]:
# ── 1b. DrugBank ID → PubChem CID crosswalk ──────────────────────────────────
pubchem = pd.read_csv(PUBCHEM_DBCHEBI_PATH)
pubchem_DB = pubchem[~pubchem['DRUGBANK_ID'].isna()]
DB2PC_dict = dict(zip(pubchem_DB['DRUGBANK_ID'], pubchem_DB['ID']))  # {DB_ID: PubChem_CID}
print(f"DrugBank -> PubChem CID: {len(DB2PC_dict):,} entries")

/tmp/ipykernel_3066890/2826461843.py:2: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  pubchem = pd.read_csv(PUBCHEM_DBCHEBI_PATH)


DrugBank -> PubChem CID: 10,702 entries


In [7]:
# ── 1c. DrugBank SMILES and name dicts ───────────────────────────────────────
DB2Smile = pd.read_csv(DB_SMILES_PATH, sep='\t', header=None)
DB2Smile = DB2Smile[~DB2Smile[1].isna()]
DB_All_Smile_dict = dict(zip(DB2Smile[0], DB2Smile[1]))   # {DB_ID: SMILES}

DB_All = pd.read_csv(DB_ALL_PATH)
DB_All_dict = dict(zip(DB_All['drugbank_id'], DB_All['name']))  # {DB_ID: name}
print(f"DrugBank SMILES: {len(DB_All_Smile_dict):,} | DrugBank names: {len(DB_All_dict):,}")

DrugBank SMILES: 10,709 | DrugBank names: 16,575


In [8]:
# ── 1d. ENSEMBL gene crosswalk ───────────────────────────────────────────────
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))  # {symbol: ENS_ID}
del ENS_2NCBI
print(f"Symbol -> Ensembl: {len(NCBI_2ENS__dict):,}")

Symbol -> Ensembl: 40,940


In [9]:
# ── 1e. NCBI Human gene_info ──────────────────────────────────────────────────
NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict  = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))
NCBI_ALL_GENE_Alias_dict = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['Synonyms']))
print(f"NCBI Human genes: {len(NCBI_ALL_GENEname_dict):,}")

NCBI Human genes: 193,352


In [10]:
# ── 1f. Gene synonym map — built ONCE (BUG FIX: original rebuilt this 6 times)
# Maps any gene alias/synonym -> canonical NCBI Symbol
synonym_map = {}
for _, row in NCBI_ALL_GENE.iterrows():
    for syn in row['Synonyms'].split('|'):   # Synonyms are pipe-separated
        synonym_map[syn.strip()] = row['Symbol']
print(f"Gene synonym map: {len(synonym_map):,} aliases")

Gene synonym map: 69,564 aliases


In [8]:
# ── 1g. Disease Ontology (DO) ─────────────────────────────────────────────────
DO_All_File = pd.read_csv(DO_PATH)
DOID_disname_dict  = dict(zip(DO_All_File['id'], DO_All_File['label']))   # DOID -> label
DOID_disAltID_dict = dict(zip(DO_All_File['id'], DO_All_File['xrefs']))   # DOID -> xrefs
print(f"DO entries: {len(DOID_disname_dict):,}")

DO entries: 11,804


In [24]:
# ── 1h. UniProt RecName ───────────────────────────────────────────────────────
Uniprot_Recname = pd.read_csv(UNIPROT_REC_PATH)
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))
del Uniprot_Recname
print(f"UniProt RecName: {len(Uniprot_Recname_dict):,}")

UniProt RecName: 794,151


In [10]:
# ── 1i. BTO tissue ontology: ID -> name ──────────────────────────────────────
BTO = pd.read_csv(BTO_PATH)
BTO_dict = dict(zip(BTO['ID'], BTO['name']))
print(f"BTO tissue entries: {len(BTO_dict):,}")

BTO tissue entries: 6,566


In [11]:
# ── 1j. GO term name and namespace ───────────────────────────────────────────
GO = pd.read_csv(GO_PATH)
GO_dict           = dict(zip(GO['id'], GO['name']))
GO_namespacedict  = dict(zip(GO['id'], GO['namespace']))
print(f"GO terms: {len(GO_dict):,}")

GO terms: 47,995


In [12]:
# ── 1k. HMDB metabolite: accession -> name and IUPAC ─────────────────────────
Metabolite = pd.read_csv(HMDB_PATH)
Metabolite_dict       = dict(zip(Metabolite['accession'], Metabolite['name']))
Metabolite_iupac_dict = dict(zip(Metabolite['accession'], Metabolite['iupac_name']))
del Metabolite
print(f"HMDB metabolites: {len(Metabolite_dict):,}")

HMDB metabolites: 217,920


In [13]:

import re
import pandas as pd

records = []

with open(PHENOTYPE_PATH, "r", encoding="utf-8", errors="ignore") as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        # Extract name
        name_match = re.search(
            r'name:\s*"([^"]+)"',
            line
        )

        # Extract ID
        id_match = re.search(
            r'id:\s*"([^"]+)"',
            line
        )

        if name_match and id_match:

            records.append({
                "ID": id_match.group(1),
                "Name": name_match.group(1)
            })

Phenotype_df = pd.DataFrame(records)

Phenotype_df = Phenotype_df.drop_duplicates()

print(Phenotype_df.head())

print(f"\nRows: {len(Phenotype_df):,}")
Phenotype_df

           ID                            Name
0  HP:0000001                             All
1  HP:0000002      Abnormality of body height
2  HP:0000003    Multicystic kidney dysplasia
3  HP:0000005             Mode of inheritance
4  HP:0000006  Autosomal dominant inheritance

Rows: 15,872


,ID,Name
0,HP:0000001,All
1,HP:0000002,Abnormality of body height
2,HP:0000003,Multicystic kidney dysplasia
3,HP:0000005,Mode of inheritance
4,HP:0000006,Autosomal dominant inheritance
...,...,...
15867,HP:3000075,Abnormal lingual nerve morphology
15868,HP:3000076,Abnormality of lingual tonsil
15869,HP:3000077,Abnormal mandible condylar process morphology
15870,HP:3000078,Abnormal mandible coronoid process morphology


In [14]:
# HPO ID -> Name
phenotype_dict = dict(
    zip(
        Phenotype_df["ID"],
        Phenotype_df["Name"]
    )
)

# Name -> HPO ID
hpo_name_to_id = {
    str(name).lower().strip(): hpo_id
    for hpo_id, name in zip(
        Phenotype_df["ID"],
        Phenotype_df["Name"]
    )
}

In [15]:
# ── 1l. Phenotype (HPO) ontology for SIDER side effects ──────────────────────
# phenotype = pd.read_csv(PHENOTYPE_PATH, sep='\t')
# display(phenotype)
# phenotype_dict = dict(zip(phenotype['ID'], phenotype['name']))
# print(f"Phenotype/HPO entries: {len(phenotype_dict):,}")

---
## 2 · Helper functions

In [14]:
def process_dataframe(file_path, head_type, tail_type):
    """
    Load a CKG TSV file and apply standard column renaming.
    Handles both column naming conventions (START_ID/END_ID and START_ID/END_ID with TYPE/type/source).
    Returns a DataFrame with Head, Tail, Relation, Relation_type, Source, Head_type, Tail_type columns.
    """
    df = pd.read_csv(file_path, sep='\t')
    rename_dict = {
        'START_ID':  'Head',
        'END_ID':    'Tail',
        'TYPE':      'Relation',
        'type':      'Relation_type',
        'source':    'Source'
    }
    existing = {k: v for k, v in rename_dict.items() if k in df.columns}
    df.rename(columns=existing, inplace=True)
    df['Head_type'] = head_type
    df['Tail_type'] = tail_type
    # Put key schema columns first; keep all remaining columns after
    specified = ['Head','Tail','Relation','Relation_type','Head_type','Tail_type','Source']
    existing_specified = [c for c in specified if c in df.columns]
    remaining = [c for c in df.columns if c not in existing_specified]
    return df[existing_specified + remaining]


def resolve_drug_head(df):
    """
    Resolve Drug Head: DrugBank ID -> PubChem CID (preferred).
    Keeps DrugBank ID as Head_Alt if PubChem lookup succeeds.
    Sets Head_ID_IS to 'Pubchem' or 'Drugbank' accordingly.
    NOTE: uses None not np.nan to avoid DTypePromotionError in numpy 2.0+
    """
    df = df.copy()
    df['Head_pubchem_mapped'] = df['Head'].map(DB2PC_dict).fillna(df['Head'])
    df['Head_pubchem_mapped'] = df['Head_pubchem_mapped'].astype(str).str.replace(r'\.0$', '', regex=True)
    # Swap: PubChem CID -> Head, original DrugBank -> Head_pubchem_mapped
    df[['Head', 'Head_pubchem_mapped']] = df[['Head_pubchem_mapped', 'Head']]
    df['Head_ID_IS'] = np.where(
        df['Head'].isna(), None,
        np.where(df['Head'].str.startswith('DB'), 'Drugbank', 'Pubchem'))
    df['Head_IUPAC']  = df['Head'].map(Pubchem_Smile_CID_Dict).fillna(df['Head'].map(DB_All_dict))
    df['Head_Smiles'] = df['Head'].map(Pubchem_IUPAC_CID_Dict).fillna(df['Head'].map(DB_All_Smile_dict))
    return df

def resolve_drug_tail(df):
    """
    Resolve Drug Tail: DrugBank ID -> PubChem CID (preferred).
    Keeps DrugBank ID as Tail_Alt if PubChem lookup succeeds.
    Sets Tail_ID_IS to 'Pubchem' or 'Drugbank' accordingly.
    NOTE: uses None not np.nan to avoid DTypePromotionError in numpy 2.0+
    """
    df = df.copy()
    df['Tail_pubchem_mapped'] = df['Tail'].map(DB2PC_dict).fillna(df['Tail'])
    df['Tail_pubchem_mapped'] = df['Tail_pubchem_mapped'].astype(str).str.replace(r'\.0$', '', regex=True)
    # Swap: PubChem CID -> Tail, original DrugBank -> Tail_pubchem_mapped
    df[['Tail', 'Tail_pubchem_mapped']] = df[['Tail_pubchem_mapped', 'Tail']]
    df['Tail_ID_IS'] = np.where(
        df['Tail'].isna(), None,
        np.where(df['Tail'].str.startswith('DB'), 'Drugbank', 'Pubchem'))
    df['Tail_IUPAC']  = df['Tail'].map(Pubchem_Smile_CID_Dict).fillna(df['Tail'].map(DB_All_dict))
    df['Tail_Smiles'] = df['Tail'].map(Pubchem_IUPAC_CID_Dict).fillna(df['Tail'].map(DB_All_Smile_dict))
    return df

def resolve_gene_tail(df):
    """
    Resolve Gene Tail: apply synonym map -> canonical Symbol -> NCBI description + Ensembl ID.
    Swaps canonical symbol into Tail, keeps original as Tail_Alias_mapped.
    """
    df = df.copy()
    df['Tail_Alias_mapped'] = df['Tail'].apply(lambda x: synonym_map.get(x, x))
    df['Tail_Detail_name']  = df['Tail_Alias_mapped'].map(NCBI_ALL_GENEname_dict)
    df['Tail_ENS']          = df['Tail_Alias_mapped'].map(NCBI_2ENS__dict)
    df[['Tail', 'Tail_Alias_mapped']] = df[['Tail_Alias_mapped', 'Tail']]
    df.rename(columns={'Tail_ENS': "Tail_Ensemble_ID's"}, inplace=True)
    df['Tail_ID_IS'] = 'NCBI_Symbol'
    return df


def resolve_gene_head(df):
    """
    Resolve Gene Head: synonym map -> canonical Symbol -> NCBI description + Ensembl ID.
    Swaps canonical symbol into Head, keeps original as Head_Alias_mapped.
    """
    df = df.copy()
    df['Head_Alias_mapped'] = df['Head'].apply(lambda x: synonym_map.get(x, x))
    df['Head_Detail_name']  = df['Head_Alias_mapped'].map(NCBI_ALL_GENEname_dict)
    df['Head_ENS']          = df['Head_Alias_mapped'].map(NCBI_2ENS__dict)
    df[['Head', 'Head_Alias_mapped']] = df[['Head_Alias_mapped', 'Head']]
    df['Head_ID_IS'] = 'NCBI Gene Symbol'
    return df


def select_cols(df, desired):
    """Select only columns that exist in df from desired list."""
    return df[[c for c in desired if c in df.columns]]


def save(df, filename):
    """Save to OUT_PATH and print confirmation."""
    path = os.path.join(OUT_PATH, filename)
    df.to_csv(path, index=False)
    print(f"Saved {len(df):,} rows -> {path}")


print("Helper functions defined.")

Helper functions defined.


---
## File 1 — Drug → Gene (DGIdb)

Source: `dgidb_targets.tsv`

In [ ]:
df1 = process_dataframe(
    os.path.join(CKG_DB_PATH, 'dgidb_targets.tsv'),
    head_type='Drug', tail_type='Gene')

df1 = resolve_drug_head(df1)
df1 = resolve_gene_tail(df1)
df1['KG_Source'] = 'CKG'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alias_mapped","Head_Alt_name","Head_Detail_name","Head_IUPAC","Head_Smiles","Head_ENS",
           "Tail_Detail_name","Tail_Alias_mapped","Tail_DO_Alt_ids","Tail_Alt_name","Head_ID_IS","Tail_ID_IS",
           "evidence_type","score","diseaseType"]
df1 = select_cols(df1, desired)
save(df1, 'File_1_Drug_Gene_CKG.csv')
del df1

---
## Files 2 & 3 — Drug → Mutation_Variant (CGI / OncoKB)

In [ ]:
def process_drug_mutation(file_path, out_filename):
    """Shared pipeline for Drug->Mutation_Variant files (Files 2 and 3)."""
    df = process_dataframe(file_path, head_type='Drug', tail_type='Mutation_Variant')
    df = resolve_drug_head(df)
    df.rename(columns={'Head_pubchem_mapped': "Head_Alt_ID's"}, inplace=True, errors='ignore')
    df['KG_Source'] = 'CKG'
    df['Tail_ID_IS'] = 'protein_mutation_name'
    desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
               "Head_Alt_ID's","Head_IUPAC","Head_Smiles","Head_ID_IS","Tail_ID_IS","evidence","response","disease"]
    df = select_cols(df, desired)
    save(df, out_filename)

process_drug_mutation(
    os.path.join(CKG_DB_PATH, 'cgi_targets_clinically_relevant_variant.tsv'),
    'File_2_Drug_MutationVariant_CKG.csv')

process_drug_mutation(
    os.path.join(CKG_DB_PATH, 'oncokb_targets_clinically_relevant_variant.tsv'),
    'File_3_Drug_MutationVariant_OncoKB_CKG.csv')

---
## File 4 — Drug → Gene (OncoKB targets)

In [ ]:
df4 = process_dataframe(
    os.path.join(CKG_DB_PATH, 'oncokb_targets.tsv'),
    head_type='Drug', tail_type='Gene')

# Remove non-gene Tail values (e.g. 'Other Biomarkers')
df4 = df4[~df4['Tail'].str.contains('Other Biomarkers', na=False)]

df4 = resolve_drug_head(df4)
df4 = resolve_gene_tail(df4)
df4.rename(columns={'Head_pubchem_mapped': "Head_Alt_ID's"}, inplace=True, errors='ignore')
df4['KG_Source'] = 'CKG'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_ID's","Head_IUPAC","Head_Smiles","Tail_Alias_mapped","Tail_Detail_name","Tail_Ensemble_ID's",
           "Head_ID_IS","Tail_ID_IS","evidence","response","disease"]
df4 = select_cols(df4, desired)
save(df4, 'File_4_Drug_Gene_OncoKB_CKG.csv')
del df4

---
## Files 8, 9, 10 — GWAS files → **DROPPED**

GWAS study, trait, and variant files are not relevant to the EvoAge project scope. Skipped entirely.

---
## File 6 — Protein_BioM → Disease (protein biomarker of disease)

In [ ]:
# /storage/Arushi/090526_EvoAge/kg_formation/data_collection/ckg/data/imports/curated
# /storage/Arushi/090526_EvoAge/kg_formation/data_collection/ckg/docker_data/neo4j/exports/nodes/relations/curated/protein_is_biomarker_of_disease.tsv'


In [97]:
df6 = process_dataframe(
    os.path.joinBASE_PATH, 'ckg/data/imports/curated/protein_is_biomarker_of_disease.tsv'),
    head_type='Protein_BioM', tail_type='Disease')

df6['Tail_DO_name']    = df6['Tail'].map(DOID_disname_dict).fillna(df6['Tail'])
# BUG FIX: original had "ail_DO_Alt_ids" (missing T) — silently dropped the column
df6['Tail_DO_Alt_ids'] = df6['Tail'].map(DOID_disAltID_dict).fillna(df6['Tail'])
df6['Head_Alt_name']   = df6['Head'].map(Uniprot_Recname_dict).fillna(df6['Head'])
df6['KG_Source']  = 'CKG'
df6['Head_ID_IS'] = 'Uniprot'
df6['Tail_ID_IS'] = 'DiseaseOntology'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_DO_Alt_ids","Tail_DO_name","Head_ID_IS","Tail_ID_IS","evidence","response","disease"]
df6 = select_cols(df6, desired)
df6 = df6.drop_duplicates()
save(df6, 'File_6_ProtBioM_Disease_CKG.csv')
del df6

IndentationError: unexpected indent (685634317.py, line 3)

---
## File 7 — Protein_QC_BioM → Tissue (QC biomarker in tissue)

In [ ]:
CKG_DOC_PATH

In [96]:
df7 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/curated/protein_is_qcmarker_in_tissue.tsv'),
    head_type='Protein_QC_BioM', tail_type='Tissue_Ontology')

df7['Tail_BTO_name'] = df7['Tail'].map(BTO_dict).fillna(df7['Tail'])
df7['Head_Alt_name'] = df7['Head'].map(Uniprot_Recname_dict).fillna(df7['Head'])
df7['KG_Source']  = 'CKG'
df7['Head_ID_IS'] = 'Uniprot'
df7['Tail_ID_IS'] = 'BrendaTO'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_BTO_name","Head_ID_IS","Tail_ID_IS"]
df7 = select_cols(df7, desired)
save(df7, 'File_7_ProtQC_bioM_Tissue_CKG.csv')
df7

Saved 255 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_7_ProtQC_bioM_Tissue_CKG.csv


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_Alt_name,Tail_BTO_name,Head_ID_IS,Tail_ID_IS
0,P69905,IS_QCMARKER_IN,BTO:0000131,Protein_QC_BioM,Tissue_Ontology,CKG,Hemopressin {ECO:0000303|PubMed:18077343},blood plasma,Uniprot,BrendaTO
1,P68871,IS_QCMARKER_IN,BTO:0000131,Protein_QC_BioM,Tissue_Ontology,CKG,Spinorphin,blood plasma,Uniprot,BrendaTO
2,P00915,IS_QCMARKER_IN,BTO:0000131,Protein_QC_BioM,Tissue_Ontology,CKG,Carbonic anhydrase 1,blood plasma,Uniprot,BrendaTO
3,E5RHP7,IS_QCMARKER_IN,BTO:0000131,Protein_QC_BioM,Tissue_Ontology,CKG,E5RHP7,blood plasma,Uniprot,BrendaTO
4,E5RH81,IS_QCMARKER_IN,BTO:0000131,Protein_QC_BioM,Tissue_Ontology,CKG,E5RH81,blood plasma,Uniprot,BrendaTO
...,...,...,...,...,...,...,...,...,...,...
250,P04070,IS_QCMARKER_IN,BTO:0000131,Protein_QC_BioM,Tissue_Ontology,CKG,Activation peptide,blood plasma,Uniprot,BrendaTO
251,E7END6,IS_QCMARKER_IN,BTO:0000131,Protein_QC_BioM,Tissue_Ontology,CKG,E7END6,blood plasma,Uniprot,BrendaTO
252,P04070-2,IS_QCMARKER_IN,BTO:0000131,Protein_QC_BioM,Tissue_Ontology,CKG,P04070-2,blood plasma,Uniprot,BrendaTO
253,P02776,IS_QCMARKER_IN,BTO:0000131,Protein_QC_BioM,Tissue_Ontology,CKG,"Platelet factor 4, short form",blood plasma,Uniprot,BrendaTO


In [ ]:
del df7


In [106]:
# # Read raw file
# df = pd.read_csv(
#     "/storage/Arushi/Gaurav_arushi_home/advik/neo4j/exports/relations/VARIANT_FOUND_IN_GENE.tsv",
#     sep=r',\s*',
#     engine='python',
#     header=0,
#     index_col=False
# )

# # Move index back into dataframe
# df.reset_index(inplace=True)
# df

---
## File 11 — Mutation_Variant → Gene

In [107]:
df11 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Gene_known_variant_found_in_gene.tsv'),
    head_type='Mutation_Variant', tail_type='Gene')

# Remove placeholder '-' gene symbols
df11 = df11[df11['Tail'] != '-']

df11 = resolve_gene_tail(df11)

# Additional ENS fallback: try original Tail if alias-mapped Tail still has no ENS
mask = df11["Tail_Ensemble_ID's"].isna()
df11.loc[mask, "Tail_Ensemble_ID's"] = df11.loc[mask, 'Tail_Alias_mapped'].map(NCBI_2ENS__dict)

df11['KG_Source']  = 'CKG'
df11['Head_ID_IS'] = 'Mutation'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_Detail_name","Tail_Ensemble_ID's","Head_ID_IS","Tail_ID_IS"]
df11 = select_cols(df11, desired)
save(df11, 'File_11_Mutation_Gene.csv')
df11

Saved 47,154,813 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_11_Mutation_Gene.csv


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Tail_Detail_name,Tail_Ensemble_ID's,Head_ID_IS,Tail_ID_IS
0,chr17:g.30185627A>G,VARIANT_FOUND_IN_GENE,NSRP1,Mutation_Variant,Gene,UniProt,CKG,nuclear speckle splicing regulatory protein 1,ENSG00000126653,Mutation,NCBI_Symbol
1,chr17:g.30185534C>A,VARIANT_FOUND_IN_GENE,NSRP1,Mutation_Variant,Gene,UniProt,CKG,nuclear speckle splicing regulatory protein 1,ENSG00000126653,Mutation,NCBI_Symbol
2,chr10:g.60784806A>T,VARIANT_FOUND_IN_GENE,CDK1,Mutation_Variant,Gene,UniProt,CKG,cyclin dependent kinase 1,ENSG00000170312,Mutation,NCBI_Symbol
3,chr6:g.8429967A>G,VARIANT_FOUND_IN_GENE,SLC35B3,Mutation_Variant,Gene,UniProt,CKG,solute carrier family 35 member B3,ENSG00000124786,Mutation,NCBI_Symbol
4,chr6:g.8429914T>C,VARIANT_FOUND_IN_GENE,SLC35B3,Mutation_Variant,Gene,UniProt,CKG,solute carrier family 35 member B3,ENSG00000124786,Mutation,NCBI_Symbol
...,...,...,...,...,...,...,...,...,...,...,...
47459590,chr7:g.33062623T>G,VARIANT_FOUND_IN_GENE,NT5C3A,Mutation_Variant,Gene,UniProt,CKG,"5'-nucleotidase, cytosolic IIIA",ENSG00000122643,Mutation,NCBI_Symbol
47459591,chr1:g.204987426G>A,VARIANT_FOUND_IN_GENE,NFASC,Mutation_Variant,Gene,UniProt,CKG,neurofascin,ENSG00000163531,Mutation,NCBI_Symbol
47459592,chr7:g.33021332T>C,VARIANT_FOUND_IN_GENE,NT5C3A,Mutation_Variant,Gene,UniProt,CKG,"5'-nucleotidase, cytosolic IIIA",ENSG00000122643,Mutation,NCBI_Symbol
47459593,chr7:g.33024094A>C,VARIANT_FOUND_IN_GENE,NT5C3A,Mutation_Variant,Gene,UniProt,CKG,"5'-nucleotidase, cytosolic IIIA",ENSG00000122643,Mutation,NCBI_Symbol


In [108]:
del df11

---
## File 12 — Protein → BiologicalProcess (GO)

In [109]:
df12 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Biological_process_associated_with.tsv'),
    head_type='Protein', tail_type='Ontology')

# Resolve GO namespace and filter out header artefacts and unmapped GO terms
df12['tail_type']       = df12['Tail'].map(GO_namespacedict)
df12['tail_detail_name']= df12['Tail'].map(GO_dict)
df12 = df12[~df12['Tail'].str.contains('TYPE', na=False)]
df12 = df12[~df12['tail_type'].isna()]

df12['Head_Alt_name'] = df12['Head'].map(Uniprot_Recname_dict).fillna(df12['Head'])
df12.rename(columns={'evidence_type': 'evidence'}, inplace=True, errors='ignore')
df12['KG_Source']  = 'CKG'
df12['Head_ID_IS'] = 'Uniprot'
df12['Tail_ID_IS'] = 'QuickGO'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_GO_name","Head_ID_IS","Tail_ID_IS","evidence","score","tail_detail_name"]
df12 = select_cols(df12, desired)
save(df12, 'File_12_Protein_BiologicalProcess_CKG.csv')
df12

/tmp/ipykernel_1970976/1938194399.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep='\t')


Saved 369,710 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_12_Protein_BiologicalProcess_CKG.csv


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_Alt_name,Head_ID_IS,Tail_ID_IS,evidence,score,tail_detail_name
0,Q96SQ7,ASSOCIATED_WITH,GO:0045603,Protein,Ontology,UniProt,CKG,Transcription factor ATOH8 {ECO:0000305},Uniprot,QuickGO,IDA,5,positive regulation of endothelial cell differ...
1,P21359,ASSOCIATED_WITH,GO:0021897,Protein,Ontology,UniProt,CKG,Neurofibromin truncated,Uniprot,QuickGO,ISS,5,forebrain astrocyte development
2,O95816,ASSOCIATED_WITH,GO:0006457,Protein,Ontology,UniProt,CKG,BAG family molecular chaperone regulator 2,Uniprot,QuickGO,TAS,5,protein folding
3,P35638,ASSOCIATED_WITH,GO:0001955,Protein,Ontology,UniProt,CKG,DNA damage-inducible transcript 3 protein,Uniprot,QuickGO,IEA,5,blood vessel maturation
4,P15336,ASSOCIATED_WITH,GO:0072740,Protein,Ontology,UniProt,CKG,Cyclic AMP-dependent transcription factor ATF-2,Uniprot,QuickGO,IEA,5,cellular response to anisomycin
...,...,...,...,...,...,...,...,...,...,...,...,...,...
369868,Q92835,ASSOCIATED_WITH,GO:0045659,Protein,Ontology,UniProt,CKG,"Phosphatidylinositol 3,4,5-trisphosphate 5-pho...",Uniprot,QuickGO,IBA,5,negative regulation of neutrophil differentiation
369869,Q9NZC2,ASSOCIATED_WITH,GO:0048143,Protein,Ontology,UniProt,CKG,Triggering receptor expressed on myeloid cells 2,Uniprot,QuickGO,ISS,5,astrocyte activation
369870,URS0000663C66_9606,ASSOCIATED_WITH,GO:0035195,Protein,Ontology,UniProt,CKG,URS0000663C66_9606,Uniprot,QuickGO,IEA,5,miRNA-mediated post-transcriptional gene silen...
369871,P08319,ASSOCIATED_WITH,GO:0006081,Protein,Ontology,UniProt,CKG,All-trans-retinol dehydrogenase [NAD(+)] ADH4 ...,Uniprot,QuickGO,IDA,5,aldehyde metabolic process


In [110]:
del df12

---
## File 13 — PMID → Cellular_component (publication mention)

In [93]:
df13 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Cellular_component_Publication_mentioned_in_publication.tsv'),
    head_type='PMID', tail_type='Cellular_component')

# Prefix PubMed IDs with 'PMID:' for namespace clarity
df13['Head'] = 'PMID:' + df13['Head'].astype(str)
df13['Tail_GO_name'] = df13['Tail'].map(GO_dict)
df13 = df13[~df13['Tail_GO_name'].isna()]
df13['KG_Source']  = 'CKG'
df13['Head_ID_IS'] = 'PubmedID'
df13['Tail_ID_IS'] = 'QuickGO'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_GO_name","Head_ID_IS","Tail_ID_IS","evidence","score"]
df13 = select_cols(df13, desired)
save(df13, 'File_13_PMID_Cellular_component_CKG.csv')
df13

Saved 87,097,962 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_13_PMID_Cellular_component_CKG.csv


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Tail_GO_name,Head_ID_IS,Tail_ID_IS
0,PMID:20339535,MENTIONED_IN_PUBLICATION,GO:0000015,PMID,Cellular_component,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO
1,PMID:20067621,MENTIONED_IN_PUBLICATION,GO:0000015,PMID,Cellular_component,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO
2,PMID:14613499,MENTIONED_IN_PUBLICATION,GO:0000015,PMID,Cellular_component,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO
3,PMID:18523666,MENTIONED_IN_PUBLICATION,GO:0000015,PMID,Cellular_component,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO
4,PMID:19492051,MENTIONED_IN_PUBLICATION,GO:0000015,PMID,Cellular_component,CKG,phosphopyruvate hydratase complex,PubmedID,QuickGO
...,...,...,...,...,...,...,...,...,...
87935640,PMID:20724623,MENTIONED_IN_PUBLICATION,GO:1990973,PMID,Cellular_component,CKG,transmembrane actin-associated (TAN) line,PubmedID,QuickGO
87935641,PMID:15815710,MENTIONED_IN_PUBLICATION,GO:1990973,PMID,Cellular_component,CKG,transmembrane actin-associated (TAN) line,PubmedID,QuickGO
87935642,PMID:14719621,MENTIONED_IN_PUBLICATION,GO:1990973,PMID,Cellular_component,CKG,transmembrane actin-associated (TAN) line,PubmedID,QuickGO
87935643,PMID:11466488,MENTIONED_IN_PUBLICATION,GO:1990973,PMID,Cellular_component,CKG,transmembrane actin-associated (TAN) line,PubmedID,QuickGO


---
## File 14 — Protein → Cellular_component

In [115]:
df14 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Cellular_component_associated_with.tsv'),
    head_type='Protein', tail_type='Cellular_component')

df14['Tail_GO_name'] = df14['Tail'].map(GO_dict)
df14 = df14[~df14['Tail_GO_name'].isna()]
df14['Head_Alt_name'] = df14['Head'].map(Uniprot_Recname_dict)
df14 = df14[~df14['Tail_GO_name'].isna()]
df14 = df14[~df14['Head_Alt_name'].isna()]

df14['KG_Source']  = 'CKG'
df14['Head_ID_IS'] = 'Uniprot_ID'
df14['Tail_ID_IS'] = 'QuickGO'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_GO_name","Head_ID_IS","Tail_ID_IS","evidence","score"]
df14 = select_cols(df14, desired)
save(df14, 'File_14_Protein_Cellular_component_CKG.csv')


/tmp/ipykernel_1970976/1938194399.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep='\t')


Saved 288,342 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_14_Protein_Cellular_component_CKG.csv


In [116]:
df14

,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_Alt_name,Tail_GO_name,Head_ID_IS,Tail_ID_IS,score
0,P55854,ASSOCIATED_WITH,GO:0005654,Protein,Cellular_component,UniProt,CKG,Small ubiquitin-related modifier 3 {ECO:0000305},nucleoplasm,Uniprot_ID,QuickGO,5
2,P32927,ASSOCIATED_WITH,GO:0009897,Protein,Cellular_component,UniProt,CKG,Cytokine receptor common subunit beta,external side of plasma membrane,Uniprot_ID,QuickGO,5
3,Q9UBF8,ASSOCIATED_WITH,GO:0000139,Protein,Cellular_component,UniProt,CKG,Phosphatidylinositol 4-kinase beta {ECO:0000305},Golgi membrane,Uniprot_ID,QuickGO,5
4,A8TX70,ASSOCIATED_WITH,GO:0005576,Protein,Cellular_component,UniProt,CKG,Collagen alpha-5(VI) chain,extracellular region,Uniprot_ID,QuickGO,5
5,Q9GZZ6,ASSOCIATED_WITH,GO:0043005,Protein,Cellular_component,UniProt,CKG,Neuronal acetylcholine receptor subunit alpha-10,neuron projection,Uniprot_ID,QuickGO,5
...,...,...,...,...,...,...,...,...,...,...,...,...
317802,A4GXA9,ASSOCIATED_WITH,GO:0048476,Protein,Cellular_component,UniProt,CKG,Probable crossover junction endonuclease EME2,Holliday junction resolvase complex,Uniprot_ID,QuickGO,5
317803,P16066,ASSOCIATED_WITH,GO:0005886,Protein,Cellular_component,UniProt,CKG,Atrial natriuretic peptide receptor 1 {ECO:000...,plasma membrane,Uniprot_ID,QuickGO,5
317804,P49754,ASSOCIATED_WITH,GO:0016020,Protein,Cellular_component,UniProt,CKG,Vacuolar protein sorting-associated protein 41...,membrane,Uniprot_ID,QuickGO,5
317805,O43464,ASSOCIATED_WITH,GO:0032991,Protein,Cellular_component,UniProt,CKG,"Serine protease HTRA2, mitochondrial",protein-containing complex,Uniprot_ID,QuickGO,5


In [113]:
del df14

---
## File 15 — Chromosome file → **DROPPED**

Chromosome node type is not included in EvoAge KG scope.

---
## File 16 — PMID → Disease

In [94]:
df16 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Disease_Publication_mentioned_in_publication.tsv'),
    head_type='PMID', tail_type='Disease')

df16['Head'] = 'PMID:' + df16['Head'].astype(str)
df16['Tail_DO_name']    = df16['Tail'].map(DOID_disname_dict)
df16['Tail_DO_Alt_ids'] = df16['Tail'].map(DOID_disAltID_dict)
df16 = df16[~df16['Tail_DO_name'].isna()]
df16['KG_Source']  = 'CKG'
df16['Head_ID_IS'] = 'PubmedID'
df16['Tail_ID_IS'] = 'DiseaseOntology'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_DO_name","Tail_DO_Alt_ids","Head_ID_IS","Tail_ID_IS","evidence","score"]
df16 = select_cols(df16, desired)
save(df16, 'File_16_PMID_Disease_CKG.csv')

Saved 192,843,423 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_16_PMID_Disease_CKG.csv


In [95]:
del df16

---
## File 17 — Metabolite → PMID

In [98]:
df17 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Metabolite_Publication_mentioned_in_publication.tsv'),
    head_type='Metabolite', tail_type='PMID')

df17['Tail'] = 'PMID:' + df17['Tail'].astype(str)
df17['Head_common_name'] = df17['Head'].map(Metabolite_dict)
df17['Head_IUPAC_name']  = df17['Head'].map(Metabolite_iupac_dict)
df17['KG_Source']  = 'CKG'
df17['Head_ID_IS'] = 'HMDB_ID'
df17['Tail_ID_IS'] = 'PubmedID'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_common_name","Head_IUPAC_name","Head_ID_IS","Tail_ID_IS","evidence","score"]
df17 = select_cols(df17, desired)
save(df17, 'File_17_Metabolite_pubmed_CKG.csv')
df17

Saved 1,031,122 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_17_Metabolite_pubmed_CKG.csv


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_common_name,Head_IUPAC_name,Head_ID_IS,Tail_ID_IS
0,HMDB0000001,MENTIONED_IN_PUBLICATION,PMID:1819935,Metabolite,PMID,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,HMDB_ID,PubmedID
1,HMDB0000001,MENTIONED_IN_PUBLICATION,PMID:2912013,Metabolite,PMID,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,HMDB_ID,PubmedID
2,HMDB0000001,MENTIONED_IN_PUBLICATION,PMID:7762816,Metabolite,PMID,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,HMDB_ID,PubmedID
3,HMDB0000001,MENTIONED_IN_PUBLICATION,PMID:9478024,Metabolite,PMID,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,HMDB_ID,PubmedID
4,HMDB0000001,MENTIONED_IN_PUBLICATION,PMID:15899597,Metabolite,PMID,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,HMDB_ID,PubmedID
...,...,...,...,...,...,...,...,...,...,...,...
1031117,HMDB0304935,MENTIONED_IN_PUBLICATION,PMID:32927015,Metabolite,PMID,HMDB,CKG,4-methoxyindoxyl sulfate,"(5-methoxy-2,3-dihydro-1H-indol-3-yl)oxidanesu...",HMDB_ID,PubmedID
1031118,HMDB0304936,MENTIONED_IN_PUBLICATION,PMID:32927015,Metabolite,PMID,HMDB,CKG,5-hydroxyindoxyl sulfate,5-methyl-1H-indole-3-sulfonic acid,HMDB_ID,PubmedID
1031119,HMDB0304937,MENTIONED_IN_PUBLICATION,PMID:32927015,Metabolite,PMID,HMDB,CKG,6-hydroxyindoxyl sulfate,(6-methyl-1H-indol-3-yl)oxidanesulfonic acid,HMDB_ID,PubmedID
1031120,HMDB0304953,MENTIONED_IN_PUBLICATION,PMID:22770225,Metabolite,PMID,HMDB,CKG,o-phenolsulfonic acid,2-hydroxybenzene-1-sulfonic acid,HMDB_ID,PubmedID


In [99]:
del df17

---
## File 18 — Protein → Molecular Function (GO)

In [118]:
df18 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Molecular_function_associated_with.tsv'),
    head_type='Protein', tail_type='Ontology')

df18['tail_type']        = df18['Tail'].map(GO_namespacedict)
df18['tail_detail_name'] = df18['Tail'].map(GO_dict)
df18['Tail_GO_name']     = df18['tail_detail_name']
df18 = df18[~df18['tail_type'].isna()]
df18 = df18[~df18['Tail_GO_name'].isna()]
df18['Head_Alt_name'] = df18['Head'].map(Uniprot_Recname_dict)
df18 = df18[~df18['Head_Alt_name'].isna()]
df18['KG_Source']  = 'CKG'
df18['Head_ID_IS'] = 'Uniprot_ID'
df18['Tail_ID_IS'] = 'Ontology'


desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_common_name","Head_Alt_name","Tail_GO_name","Head_ID_IS","Tail_ID_IS",
           "evidence_type","score","tail_detail_name"]
df18 = select_cols(df18, desired)
save(df18, 'File_18_protein_MolecularFunction_CKG.csv')
df18

/tmp/ipykernel_1970976/1938194399.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep='\t')


Saved 220,704 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_18_protein_MolecularFunction_CKG.csv


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_Alt_name,Tail_GO_name,Head_ID_IS,Tail_ID_IS,evidence_type,score,tail_detail_name
0,O94778,ASSOCIATED_WITH,GO:0015265,Protein,Ontology,UniProt,CKG,Aquaporin-8 {ECO:0000303|PubMed:34292591},urea channel activity,Uniprot_ID,Ontology,ISS,5,urea channel activity
1,Q9NQ69,ASSOCIATED_WITH,GO:0003677,Protein,Ontology,UniProt,CKG,LIM/homeobox protein Lhx9,DNA binding,Uniprot_ID,Ontology,IEA,5,DNA binding
2,P17302,ASSOCIATED_WITH,GO:0022857,Protein,Ontology,UniProt,CKG,Gap junction alpha-1 protein,transmembrane transporter activity,Uniprot_ID,Ontology,IEA,5,transmembrane transporter activity
3,Q9HCE7,ASSOCIATED_WITH,GO:0070411,Protein,Ontology,UniProt,CKG,E3 ubiquitin-protein ligase SMURF1,I-SMAD binding,Uniprot_ID,Ontology,IPI,5,I-SMAD binding
4,Q9UMS4,ASSOCIATED_WITH,GO:0042802,Protein,Ontology,UniProt,CKG,Pre-mRNA-processing factor 19 {ECO:0000305},identical protein binding,Uniprot_ID,Ontology,IPI,5,identical protein binding
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242607,Q99795,ASSOCIATED_WITH,GO:0005515,Protein,Ontology,UniProt,CKG,Cell surface A33 antigen,protein binding,Uniprot_ID,Ontology,IPI,5,protein binding
242609,Q13423,ASSOCIATED_WITH,GO:0051287,Protein,Ontology,UniProt,CKG,"NAD(P) transhydrogenase, mitochondrial",NAD binding,Uniprot_ID,Ontology,TAS,5,NAD binding
242612,O00418,ASSOCIATED_WITH,GO:0004672,Protein,Ontology,UniProt,CKG,Eukaryotic elongation factor 2 kinase,protein kinase activity,Uniprot_ID,Ontology,TAS,5,protein kinase activity
242613,Q7Z3C6,ASSOCIATED_WITH,GO:0005515,Protein,Ontology,UniProt,CKG,Autophagy-related protein 9A {ECO:0000303|PubM...,protein binding,Uniprot_ID,Ontology,IPI,5,protein binding


---
## File 19 — Peptide_belongs_to_proteinseq → **DROPPED**

Peptide sequence data is not suitable for KG integration.

---
## File 20 — Protein → Cellular_component (integrated)

In [25]:
BASE_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/data_collection/'

In [27]:
df20 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Protein_Cellular_component_associated_with_integrated.tsv'),
    head_type='Protein', tail_type='Cellular_component')

df20['Head_Alt_name'] = df20['Head'].map(Uniprot_Recname_dict)
df20['Tail_GO_name']  = df20['Tail'].map(GO_dict)
df20 = df20[~df20['Tail_GO_name'].isna()]
df20['KG_Source']  = 'CKG'
df20['Head_ID_IS'] = 'Uniprot_ID'
df20['Tail_ID_IS'] = 'QuickGO'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_common_name","Head_Alt_name","Tail_GO_name","Head_ID_IS","Tail_ID_IS","evidence_type","score"]
df20 = select_cols(df20, desired)
save(df20, 'File_20_protein_cellular_compartment_CKG.csv')
del df20

Saved 20,555,035 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_20_protein_cellular_compartment_CKG.csv


---
## File 21 — Protein → Disease (integrated, ICD filtered)

In [29]:
df21 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Protein_Disease_associated_with_integrated.tsv'),
    head_type='Protein', tail_type='Disease')

# Remove ICD-prefixed disease IDs — not resolvable via DOID
df21 = df21[~df21['Tail'].str.startswith('ICD')]
df21['Head_Alt_name']    = df21['Head'].map(Uniprot_Recname_dict)
df21['Tail_DO_name']     = df21['Tail'].map(DOID_disname_dict)
df21['Tail_DO_Alt_ids']  = df21['Tail'].map(DOID_disAltID_dict).fillna(df21['Tail'])
df21 = df21[~df21['Tail_DO_name'].isna()]
df21['KG_Source']  = 'CKG'
df21['Head_ID_IS'] = 'Uniprot_ID'
df21['Tail_ID_IS'] = 'Disease_Ontology'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_common_name","Head_Alt_name","Tail_DO_name","Tail_DO_Alt_ids","Head_ID_IS","Tail_ID_IS",
           "evidence_type","score"]
df21 = select_cols(df21, desired)
save(df21, 'File_21_protein_Disease_CKG.csv')
# del df21
df21

Saved 30,241,580 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_21_protein_Disease_CKG.csv


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_Alt_name,Tail_DO_name,Tail_DO_Alt_ids,Head_ID_IS,Tail_ID_IS,evidence_type,score
0,Q9UCA1,ASSOCIATED_WITH_INTEGRATED,DOID:10605,Protein,Disease,DISEASES,CKG,Thrombin heavy chain,short bowel syndrome,MESH:D012778|NCI:C99059|SNOMEDCT_US_2023_03_01...,Uniprot_ID,Disease_Ontology,compiled,1.657
2,Q8N425,ASSOCIATED_WITH_INTEGRATED,DOID:654,Protein,Disease,DISEASES,CKG,(E3-independent) E2 ubiquitin-conjugating enzyme,overnutrition,MESH:D044343|SNOMEDCT_US_2023_03_01:302872003|...,Uniprot_ID,Disease_Ontology,compiled,0.982
3,Q86UJ9,ASSOCIATED_WITH_INTEGRATED,DOID:0060037,Protein,Disease,DISEASES,CKG,Cyclic AMP-responsive element-binding protein 5,developmental disorder of mental health,DOID:0060037,Uniprot_ID,Disease_Ontology,compiled,1.840
4,Q7Z639,ASSOCIATED_WITH_INTEGRATED,DOID:1240,Protein,Disease,DISEASES,CKG,Ubiquitin conjugation factor E4 A {ECO:0000305},leukemia,ICD10CM:C95.90|ICD9CM:208|ICDO:9800/3|MESH:D00...,Uniprot_ID,Disease_Ontology,compiled,1.128
5,Q1P9S9,ASSOCIATED_WITH_INTEGRATED,DOID:0050328,Protein,Disease,DISEASES,CKG,Arresten,congenital hypothyroidism,GARD:1487|ICD10CM:E00.1|ICD10CM:E03.1|ICD9CM:2...,Uniprot_ID,Disease_Ontology,compiled,0.579
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48437728,Q8TDC2,ASSOCIATED_WITH_INTEGRATED,DOID:28,Protein,Disease,DISEASES,CKG,Serine/threonine-protein kinase BRSK1,endocrine system disease,ICD10CM:E34.9|ICD9CM:259.9|MESH:D004700|NCI:C3...,Uniprot_ID,Disease_Ontology,compiled,1.253
48437729,D3DVD9,ASSOCIATED_WITH_INTEGRATED,DOID:4305,Protein,Disease,DISEASES,CKG,C-reactive protein(1-205),bone giant cell tumor,MESH:D018212|NCI:C121932|SNOMEDCT_US_2023_03_0...,Uniprot_ID,Disease_Ontology,compiled,0.750
48437730,Q96EL3,ASSOCIATED_WITH_INTEGRATED,DOID:0080001,Protein,Disease,DISEASES,CKG,Large ribosomal subunit protein mL53 {ECO:0000...,bone disease,ICD10CM:M89.9|MESH:D001847|SNOMEDCT_US_2023_03...,Uniprot_ID,Disease_Ontology,compiled,0.572
48437731,P02788,ASSOCIATED_WITH_INTEGRATED,DOID:1657,Protein,Disease,DISEASES,CKG,Lactoferroxin-C,ventricular septal defect,GARD:7853|ICD10CM:Q21.0|ICD9CM:745.4|MESH:D006...,Uniprot_ID,Disease_Ontology,compiled,0.586


In [ ]:
del df21


---
## File 23 — Protein → Tissue (BTO only)

In [119]:
df23 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Protein_Tissue_associated_with_integrated.tsv'),
    head_type='Protein', tail_type='Tissue')

# Keep only BTO-prefixed tissue IDs
df23 = df23[df23['Tail'].str.startswith('BTO')]
df23['Head_Alt_name']  = df23['Head'].map(Uniprot_Recname_dict)
df23['Tail_BTO_name']  = df23['Tail'].map(BTO_dict)
df23['KG_Source']  = 'CKG'
df23['Head_ID_IS'] = 'Uniprot_ID'
df23['Tail_ID_IS'] = 'brendaTissueid'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_BTO_name","Head_ID_IS","Tail_ID_IS","evidence_type","score"]
df23 = select_cols(df23, desired)
save(df23, 'File_23_protein_tissue_CKG.csv')
# del df23
df23

Saved 40,474,167 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_23_protein_tissue_CKG.csv


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_Alt_name,Tail_BTO_name,Head_ID_IS,Tail_ID_IS,evidence_type,score
0,Q96PA4,ASSOCIATED_WITH_INTEGRATED,BTO:0000138,Protein,Tissue,TISSUES,CKG,C-type lectin domain family 7 member A,midbrain,Uniprot_ID,brendaTissueid,compiled,1.145
1,O15267,ASSOCIATED_WITH_INTEGRATED,BTO:0001764,Protein,Tissue,TISSUES,CKG,Short stature homeobox protein,neural crest,Uniprot_ID,brendaTissueid,compiled,1.065
2,D3DNW1,ASSOCIATED_WITH_INTEGRATED,BTO:0005601,Protein,Tissue,TISSUES,CKG,Urotensin-2B,paracentral lobule,Uniprot_ID,brendaTissueid,compiled,0.517
3,Q5T4H6,ASSOCIATED_WITH_INTEGRATED,BTO:0001233,Protein,Tissue,TISSUES,CKG,Mucolipin-3 {ECO:0000303|PubMed:19497048},plant embryo,Uniprot_ID,brendaTissueid,compiled,0.821
4,Q96HE1,ASSOCIATED_WITH_INTEGRATED,BTO:0005198,Protein,Tissue,TISSUES,CKG,Neurogenin-1,adipose-derived stem cell,Uniprot_ID,brendaTissueid,compiled,0.973
...,...,...,...,...,...,...,...,...,...,...,...,...,...
41722253,Q53SJ1,ASSOCIATED_WITH_INTEGRATED,BTO:0005453,Protein,Tissue,TISSUES,CKG,Rho-related GTP-binding protein RhoQ,culture condition:CD4+ cell,Uniprot_ID,brendaTissueid,compiled,0.533
41722255,A8KAJ4,ASSOCIATED_WITH_INTEGRATED,BTO:0000956,Protein,Tissue,TISSUES,CKG,Interferon alpha/beta receptor 2,gut epithelium,Uniprot_ID,brendaTissueid,compiled,1.139
41722256,Q9HBU3,ASSOCIATED_WITH_INTEGRATED,BTO:0001491,Protein,Tissue,TISSUES,CKG,Myoferlin,viscus,Uniprot_ID,brendaTissueid,compiled,4.706
41722257,Q7L8L3,ASSOCIATED_WITH_INTEGRATED,BTO:0000675,Protein,Tissue,TISSUES,CKG,Ammonium transporter Rh type A {ECO:0000305},SW-620 cell,Uniprot_ID,brendaTissueid,compiled,1.158


In [120]:
del df23

---
## File 24 — Protein → Taxonomy (inspect only, not exported)

In [ ]:
# df24 = process_dataframe(
#     os.path.join(CKG_DOC_PATH, 'databases/Protein_belongs_to_taxonomy.tsv'),
#     head_type='Protein', tail_type='Taxanomy')
# print("Taxonomy Tail values:", set(df24['Tail']))
# # Not exported — taxonomy annotation is supplementary

---
## File 25 — Protein → Disease (DisGeNET)

In [ ]:
df25 = process_dataframe(
    os.path.join(CKG_DOC_PATH, 'databases/Protein_disgenet_associated_with.tsv'), # This file of CKG is lecenced by Disgenet now
    head_type='Protein', tail_type='Disease')

df25['Head_Alt_name']   = df25['Head'].map(Uniprot_Recname_dict)
df25['Tail_DO_name']    = df25['Tail'].map(DOID_disname_dict)
df25['Tail_DO_Alt_ids'] = df25['Tail'].map(DOID_disAltID_dict).fillna(df25['Tail'])
df25['KG_Source']  = 'CKG'
df25['Head_ID_IS'] = 'Uniprot_ID'
df25['Tail_ID_IS'] = 'Disease_Ontology'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_DO_name","Tail_DO_Alt_ids","Head_ID_IS","Tail_ID_IS","evidence_type","score"]
df25 = select_cols(df25, desired)
save(df25, 'File_25_protein_disease_CKG.csv')
del df25

---
## File 26 — Gene → Protein (translated into)

In [32]:
df26 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Protein_gene_translated_into.tsv'),  
    head_type='Gene', tail_type='Protein')

# BUG FIX: original ran the Head swap TWICE — double-swap returns data to original state
df26 = resolve_gene_head(df26)
df26['Tail_Alt_name'] = df26['Tail'].map(Uniprot_Recname_dict)
df26['KG_Source']  = 'CKG'
df26['Tail_ID_IS'] = 'Uniprot_ID'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Detail_name","Head_ENS","Tail_Alt_name","Head_ID_IS","Tail_ID_IS","evidence_type","score"]
df26 = select_cols(df26, desired)
save(df26, 'File_26_Gene_protein_CKG.csv')
# del df26
df26

Saved 596,126 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_26_Gene_protein_CKG.csv


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_Detail_name,Head_ENS,Tail_Alt_name,Head_ID_IS,Tail_ID_IS
0,ADORA2A,GENE_TRANSLATED_INTO,P29274,Gene,Protein,UniProt,CKG,adenosine A2a receptor,ENSG00000128271,Adenosine receptor A2a,NCBI Gene Symbol,Uniprot_ID
1,HTR4,GENE_TRANSLATED_INTO,Q13639,Gene,Protein,UniProt,CKG,5-hydroxytryptamine receptor 4,ENSG00000164270,5-hydroxytryptamine receptor 4,NCBI Gene Symbol,Uniprot_ID
2,CHRM1,GENE_TRANSLATED_INTO,P11229-2,Gene,Protein,UniProt,CKG,cholinergic receptor muscarinic 1,ENSG00000168539,NaN,NCBI Gene Symbol,Uniprot_ID
3,A1BG,GENE_TRANSLATED_INTO,P04217,Gene,Protein,UniProt,CKG,alpha-1-B glycoprotein,ENSG00000121410,Alpha-1B-glycoprotein,NCBI Gene Symbol,Uniprot_ID
4,SLC3A2,GENE_TRANSLATED_INTO,P08195-3,Gene,Protein,UniProt,CKG,solute carrier family 3 member 2,ENSG00000168003,NaN,NCBI Gene Symbol,Uniprot_ID
...,...,...,...,...,...,...,...,...,...,...,...,...
596121,ND6,GENE_TRANSLATED_INTO,A0A343JYI0,Gene,Protein,UniProt,CKG,NADH dehydrogenase subunit 6,NaN,NaN,NCBI Gene Symbol,Uniprot_ID
596122,HLA-C,GENE_TRANSLATED_INTO,D6C6G2,Gene,Protein,UniProt,CKG,"major histocompatibility complex, class I, C",ENSG00000204525,NaN,NCBI Gene Symbol,Uniprot_ID
596123,INS120,GENE_TRANSLATED_INTO,V9GZZ3,Gene,Protein,UniProt,CKG,NaN,NaN,NaN,NCBI Gene Symbol,Uniprot_ID
596124,HLA-DQB2,GENE_TRANSLATED_INTO,D7REY6,Gene,Protein,UniProt,CKG,"major histocompatibility complex, class II, DQ...",ENSG00000232629,NaN,NCBI Gene Symbol,Uniprot_ID


In [33]:
del df26

---
## File 27 — Protein → PDB structure

In [36]:
# df27 = process_dataframe(
#     os.path.join(BASE_PATH, 'ckg/data/imports/databases/Protein_has_structure.tsv'),
#     head_type='Protein', tail_type='PDB')

# df27['Head_Alt_name'] = df27['Head'].map(Uniprot_Recname_dict)
# df27['KG_Source']  = 'CKG'
# df27['Head_ID_IS'] = 'Uniprot_ID'
# df27['Tail_ID_IS'] = 'PDB_ID'

# desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
#            "Head_Alt_name","Tail_Alt_name","Head_ID_IS","Tail_ID_IS","evidence_type","score"]
# df27 = select_cols(df27, desired)
# save(df27, 'File_27_Protein_PDB_CKG.csv')
# # del df27
# df27

---
## File 28 — Mutation → Protein (variant found in protein)

In [124]:
132701822 - 48929492

83772330

In [37]:
df28 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Protein_known_variant_found_in_protein.tsv'),
    head_type='Mutation', tail_type='Protein')

df28['Tail_Alt_name'] = df28['Tail'].map(Uniprot_Recname_dict)
df28['KG_Source']  = 'CKG'
df28['Head_ID_IS'] = 'Mutation'
df28['Tail_ID_IS'] = 'Uniprot_ID'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_Alt_name","Head_ID_IS","Tail_ID_IS","evidence_type","score"]
df28 = select_cols(df28, desired)
save(df28, 'File_28_Mutation_Protein_CKG.csv')
df28

FileNotFoundError: [Errno 2] No such file or directory: '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/ckg/data/imports/databases/Protein_known_variant_found_in_protein.tsv'

---
## File 28_1 — Transcript → Protein (translated into)

In [38]:
# df28_1 = process_dataframe(
#     os.path.join(CKG_DOC_PATH, 'databases/Protein_transcript_translated_into.tsv'),
#     head_type='Transcript', tail_type='Protein')

# df28_1['Tail_Alt_name'] = df28_1['Tail'].map(Uniprot_Recname_dict)
# df28_1['KG_Source']  = 'CKG'
# df28_1['Head_ID_IS'] = 'Transcript'
# df28_1['Tail_ID_IS'] = 'Uniprot_ID'

# desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
#            "Head_Alt_name","Tail_Alt_name","Head_ID_IS","Tail_ID_IS","evidence_type","score"]
# df28_1 = select_cols(df28_1, desired)
# save(df28_1, 'File_28_1_Transcript_Protein_CKG.csv')
# del df28_1

---
## File 29 — Protein → Protein (transcript isoform)

In [39]:
df29 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Transcript_is_isoform.tsv'),
    head_type='Protein', tail_type='Protein')

df29['Head_Alt_name'] = df29['Head'].map(Uniprot_Recname_dict)
df29['Tail_Alt_name'] = df29['Tail'].map(Uniprot_Recname_dict)
df29['KG_Source']  = 'CKG'
df29['Head_ID_IS'] = 'Uniprot_ID'
df29['Tail_ID_IS'] = 'Uniprot_ID'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_Alt_name","Head_ID_IS","Tail_ID_IS","evidence_type","score"]
df29 = select_cols(df29, desired)
save(df29, 'File_29_Protein_Protein_CKG.csv')
df29

Saved 516,029 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_29_Protein_Protein_CKG.csv


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_Alt_name,Tail_Alt_name,Head_ID_IS,Tail_ID_IS
0,Q9H0P0-4,IS_ISOFORM,Q9H0P0,Protein,Protein,UniProt,CKG,NaN,Cytosolic 5'-nucleotidase 3A {ECO:0000305|PubM...,Uniprot_ID,Uniprot_ID
1,P49902-2,IS_ISOFORM,P49902,Protein,Protein,UniProt,CKG,NaN,Cytosolic purine 5'-nucleotidase {ECO:0000305|...,Uniprot_ID,Uniprot_ID
2,P17174-1,IS_ISOFORM,P17174,Protein,Protein,UniProt,CKG,NaN,"Aspartate aminotransferase, cytoplasmic {ECO:0...",Uniprot_ID,Uniprot_ID
3,P05067-10,IS_ISOFORM,P05067,Protein,Protein,UniProt,CKG,NaN,C31,Uniprot_ID,Uniprot_ID
4,Q5QJU3-3,IS_ISOFORM,Q5QJU3-3,Protein,Protein,UniProt,CKG,NaN,NaN,Uniprot_ID,Uniprot_ID
...,...,...,...,...,...,...,...,...,...,...,...
516024,Q03936-1,IS_ISOFORM,Q03936-2,Protein,Protein,UniProt,CKG,NaN,NaN,Uniprot_ID,Uniprot_ID
516025,Q16670-1,IS_ISOFORM,Q16670-1,Protein,Protein,UniProt,CKG,NaN,NaN,Uniprot_ID,Uniprot_ID
516026,P17040-2,IS_ISOFORM,P17040-1,Protein,Protein,UniProt,CKG,NaN,NaN,Uniprot_ID,Uniprot_ID
516027,Q9UDY2-5,IS_ISOFORM,Q9UDY2-4,Protein,Protein,UniProt,CKG,NaN,NaN,Uniprot_ID,Uniprot_ID


In [40]:
del df29

---
## File 30 — Drug → Gene (CGI targets)

In [41]:
df30 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/cgi_targets.tsv'),
    head_type='Drug', tail_type='Gene')

df30 = resolve_drug_head(df30)
df30.rename(columns={'Head_pubchem_mapped': "Head_Alt_ID's"}, inplace=True, errors='ignore')
df30['KG_Source'] = 'CKG'
df30['Tail_Alias'] = df30['Tail'].map(NCBI_ALL_GENE_Alias_dict)
df30['Tail_Name']  = df30['Tail'].map(NCBI_ALL_GENEname_dict)

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_IUPAC","Head_Smiles","Head_Alt_ID's","Tail_Alias","Tail_Alias_mapped",
           "Tail_Detail_name","Tail_Name","Tail_Ensemble_ID's","Head_ID_IS","Tail_ID_IS",
           "evidence","response","disease"]
df30 = select_cols(df30, desired)
save(df30, 'File_30_Drug_Gene_CKG.csv')
df30

Saved 799 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_30_Drug_Gene_CKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Source,KG_Source,Head_IUPAC,Head_Smiles,...,Tail_Alias,Tail_Alias_mapped,Tail_Detail_name,Tail_Name,Tail_Ensemble_ID's,Head_ID_IS,Tail_ID_IS,evidence,response,disease
0,PI3K pathway inhibitors,CURATED_TARGETS,NRAS,Drug,curated,Gene,CGI,CKG,NaN,NaN,...,ALPS4|CMNS|KRAS|N-ras|NCMS|NRAS1|NS6,KRAS,"NRAS proto-oncogene, GTPase","NRAS proto-oncogene, GTPase",ENSG00000213281,Pubchem,NCBI_Symbol,Pre-clinical,Resistant,Endometrium
1,EGFR mAb inhibitors,CURATED_TARGETS,PTEN,Drug,curated,Gene,CGI,CKG,NaN,NaN,...,10q23del|BZS|CWS1|DEC|GLM2|MHAM|MMAC1|PTEN1|PT...,PTEN,phosphatase and tensin homolog,phosphatase and tensin homolog,ENSG00000171862,Pubchem,NCBI_Symbol,Late trials,Resistant,Colorectal adenocarcinoma
2,36314,CURATED_TARGETS,TUBB3,Drug,curated,Gene,CGI,CKG,CC1=C2[C@H](C(=O)[C@@]3([C@H](C[C@@H]4[C@]([C@...,"[(1S,2S,3R,4S,7R,9S,10S,12R,15S)-4,12-diacetyl...",...,CDCBM|CDCBM1|CFEOM3|CFEOM3A|FEOM3|TUBB4|beta-4,TUBB3,tubulin beta 3 class III,tubulin beta 3 class III,ENSG00000258947,Pubchem,NCBI_Symbol,Pre-clinical,Resistant,Bladder
3,132971,CURATED_TARGETS,CYP17A1,Drug,curated,Gene,CGI,CKG,C[C@]12CC[C@@H](CC1=CC[C@@H]3[C@@H]2CC[C@]4([C...,"(3S,8R,9S,10R,13S,14S)-10,13-dimethyl-17-pyrid...",...,CPT7|CYP17|P450C17|S17AH,CYP17A1,cytochrome P450 family 17 subfamily A member 1,cytochrome P450 family 17 subfamily A member 1,ENSG00000148795,Pubchem,NCBI_Symbol,Early trials,Responsive,Prostate adenocarcinoma
4,BRAF inhibitors,CURATED_TARGETS,AKT1,Drug,curated,Gene,CGI,CKG,NaN,NaN,...,AKT|PKB|PKB-ALPHA|PRKBA|RAC|RAC-ALPHA,AKT1,AKT serine/threonine kinase 1,AKT serine/threonine kinase 1,ENSG00000142208,Pubchem,NCBI_Symbol,Case report,Resistant,Cutaneous melanoma
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
794,3062316,CURATED_TARGETS,NF1,Drug,curated,Gene,CGI,CKG,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(...,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...,...,NFNS|VRNF|WSS,NF1,neurofibromin 1,neurofibromin 1,ENSG00000196712,Pubchem,NCBI_Symbol,Pre-clinical,Resistant,Lung
795,42611257,CURATED_TARGETS,BRAF,Drug,curated,Gene,CGI,CKG,CCCS(=O)(=O)NC1=C(C(=C(C=C1)F)C(=O)C2=CNC3=C2C...,"N-[3-[5-(4-chlorophenyl)-1H-pyrrolo[2,3-b]pyri...",...,B-RAF1|B-raf|BRAF-1|BRAF1|NS7|RAFB1,BRAF,"B-Raf proto-oncogene, serine/threonine kinase","B-Raf proto-oncogene, serine/threonine kinase",ENSG00000157764,Pubchem,NCBI_Symbol,NCCN guidelines,Responsive,Cutaneous melanoma
796,216239,CURATED_TARGETS,KIT,Drug,curated,Gene,CGI,CKG,CNC(=O)C1=NC=CC(=C1)OC2=CC=C(C=C2)NC(=O)NC3=CC...,4-[4-[[4-chloro-3-(trifluoromethyl)phenyl]carb...,...,C-Kit|CD117|MASTC|PBT|SCFR,KIT,"KIT proto-oncogene, receptor tyrosine kinase","KIT proto-oncogene, receptor tyrosine kinase",ENSG00000157404,Pubchem,NCBI_Symbol,Early trials,Responsive,Gastrointestinal stromal
797,5329102,CURATED_TARGETS,KIT,Drug,curated,Gene,CGI,CKG,CCN(CC)CCNC(=O)C1=C(NC(=C1C)/C=C\2/C3=C(C=CC(=...,N-[2-(diethylamino)ethyl]-5-[(Z)-(5-fluoro-2-o...,...,C-Kit|CD117|MASTC|PBT|SCFR,KIT,"KIT proto-oncogene, receptor tyrosine kinase","KIT proto-oncogene, receptor tyrosine kinase",ENSG00000157404,Pubchem,NCBI_Symbol,Pre-clinical,Responsive,Thymic


In [42]:
del df30

---
## File 31 — Metabolite → Disease (HMDB)

In [104]:
# cols = [
#     "source_type",
#     "source_id",
#     "target_type",
#     "target_id",
#     "properties"
# ]

# df = pd.read_csv(
#     "/storage/Arushi/Gaurav_arushi_home/advik/neo4j/exports/relations/ASSOCIATED_WITH.tsv",
#     sep=",",
#     names=cols,
#     header=0,
#     engine="python",
#     usecols=[0,1,2,3,4],
#     on_bad_lines="skip"
# )
# df = df.reset_index()
# df = df.rename(columns={
#     'level_0': 'source_type',
#     'level_1': 'START_ID',
#     'source_type': 'target_type',
#     'source_id': 'END_ID',
#     'target_type': 'properties'
# })

# # Keep only required columns
# df = df[
#     ['source_type', 'START_ID', 'target_type', 'END_ID', 'properties']
# ]

# df = df.loc[:, ~df.columns.duplicated()]

# cols_to_clean = [
#     'source_type',
#     'START_ID',
#     'target_type',
#     'END_ID'
# ]

# for col in cols_to_clean:
#     df[col] = (
#         df[col]
#         .astype(str)
#         .str.replace('"', '', regex=False)
#         .str.strip()
#     )

# df

In [103]:
# df_metabolite_disease = df[
#     (df['source_type'] == 'Metabolite') &
#     (df['target_type'] == 'Disease')
# ]

# df_metabolite_disease.to_csv(
#     os.path.join(BASE_PATH, 'ckg/data/imports/databases/hmdb_associated_with_disease.tsv'),
#     sep='\t',
#     index=False
# )

In [105]:
df31 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/hmdb_associated_with_disease.tsv'),
    head_type='Metabolite', tail_type='Disease')

df31['Head_common_name'] = df31['Head'].map(Metabolite_dict)
df31['Head_IUPAC_name']  = df31['Head'].map(Metabolite_iupac_dict)
df31['Tail_DO_name']     = df31['Tail'].map(DOID_disname_dict)
df31['Tail_DO_Alt_ids']  = df31['Tail'].map(DOID_disAltID_dict).fillna(df31['Tail'])
df31['KG_Source']  = 'CKG'
df31['Head_ID_IS'] = 'HMDB'
df31['Tail_ID_IS'] = 'Disease_Ontology'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_common_name","Head_IUPAC_name","Tail_DO_name","Tail_DO_Alt_ids",
           "Head_ID_IS","Tail_ID_IS","evidence","response","disease"]
df31 = select_cols(df31, desired)
save(df31, 'File_31_Metabolite_Disease_CKG.csv')
df31

Saved 24,747 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_31_Metabolite_Disease_CKG.csv


/tmp/ipykernel_1970976/2832786286.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df31['Tail_DO_Alt_ids']  = df31['Tail'].map(DOID_disAltID_dict).fillna(df31['Tail'])


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_common_name,Head_IUPAC_name,Tail_DO_name,Tail_DO_Alt_ids,Head_ID_IS,Tail_ID_IS
0,HMDB0000001,ASSOCIATED_WITH,610247,Metabolite,Disease,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,NaN,610247,HMDB,Disease_Ontology
1,HMDB0000001,ASSOCIATED_WITH,601665,Metabolite,Disease,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,NaN,601665,HMDB,Disease_Ontology
2,HMDB0000001,ASSOCIATED_WITH,104300,Metabolite,Disease,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,NaN,104300,HMDB,Disease_Ontology
3,HMDB0000001,ASSOCIATED_WITH,125853,Metabolite,Disease,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,NaN,125853,HMDB,Disease_Ontology
4,HMDB0000001,ASSOCIATED_WITH,248600,Metabolite,Disease,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,NaN,248600,HMDB,Disease_Ontology
...,...,...,...,...,...,...,...,...,...,...,...,...,...
24742,HMDB0240219,ASSOCIATED_WITH,114500,Metabolite,Disease,HMDB,CKG,cis-vaccenic acid,(11Z)-octadec-11-enoic acid,NaN,114500,HMDB,Disease_Ontology
24743,HMDB0240252,ASSOCIATED_WITH,114500,Metabolite,Disease,HMDB,CKG,salicyluric beta-d-glucuronide,"(2S,3S,4S,5R,6S)-3,4,5-trihydroxy-6-({2-[(2-hy...",NaN,114500,HMDB,Disease_Ontology
24744,HMDB0240261,ASSOCIATED_WITH,114500,Metabolite,Disease,HMDB,CKG,lysopi(18:0/0:0),[(2R)-2-hydroxy-3-(octadecanoyloxy)propoxy]({[...,NaN,114500,HMDB,Disease_Ontology
24745,HMDB0240262,ASSOCIATED_WITH,114500,Metabolite,Disease,HMDB,CKG,lysopc(0:0/16:0),(2-{[(2R)-2-(hexadecanoyloxy)-3-hydroxypropyl ...,NaN,114500,HMDB,Disease_Ontology


---
## File 32 — Metabolite → Protein (HMDB)

In [101]:
# df_metabolite_protein = df[
#     (df['source_type'] == 'Metabolite') &
#     (df['target_type'] == 'Protein')
# ]

# df_metabolite_protein.to_csv(
#     os.path.join(BASE_PATH, 'ckg/data/imports/databases/hmdb_associated_with_protein.tsv'),
#     sep='\t',
#     index=False
# )

In [102]:
# df_metabolite_protein

In [ ]:
df32 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/hmdb_associated_with_protein.tsv'),
    head_type='Metabolite', tail_type='Protein')

df32 = resolve_drug_head(df32)

# df32['Head_common_name'] = df32['Head'].map(Metabolite_dict)
# df32['Head_IUPAC_name']  = df32['Head'].map(Metabolite_iupac_dict)
# df32['Tail_Alt_name']    = df32['Tail'].map(Uniprot_Recname_dict)
# df32['KG_Source']  = 'CKG'
# df32['Head_ID_IS'] = 'HMDB'
# df32['Tail_ID_IS'] = 'Uniprot_ID'

# desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
#            "Head_common_name","Head_IUPAC_name","Tail_Alt_name","Head_ID_IS","Tail_ID_IS",
#            "evidence","response","disease"]
# df32 = select_cols(df32, desired)
# save(df32, 'File_32_Metabolite_Protein_CKG.csv')
df32

In [19]:
df32 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/hmdb_associated_with_protein.tsv'),
    head_type='Metabolite', tail_type='Protein')

df32 = resolve_drug_head(df32)
df32

/tmp/ipykernel_2874843/1938194399.py:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Head_IUPAC']  = df['Head'].map(Pubchem_Smile_CID_Dict).fillna(df['Head'].map(DB_All_dict))
/tmp/ipykernel_2874843/1938194399.py:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Head_Smiles'] = df['Head'].map(Pubchem_IUPAC_CID_Dict).fillna(df['Head'].map(DB_All_Smile_dict))


,Head,Tail,Relation,Head_type,Tail_type,Source,Head_pubchem_mapped,Head_ID_IS,Head_IUPAC,Head_Smiles
0,HMDB0000001,Q96KN2,ASSOCIATED_WITH,Metabolite,Protein,HMDB,HMDB0000001,Pubchem,NaN,NaN
1,HMDB0000001,O60678,ASSOCIATED_WITH,Metabolite,Protein,HMDB,HMDB0000001,Pubchem,NaN,NaN
2,HMDB0000002,P49366,ASSOCIATED_WITH,Metabolite,Protein,HMDB,HMDB0000002,Pubchem,NaN,NaN
3,HMDB0000002,P52788,ASSOCIATED_WITH,Metabolite,Protein,HMDB,HMDB0000002,Pubchem,NaN,NaN
4,HMDB0000002,P17707,ASSOCIATED_WITH,Metabolite,Protein,HMDB,HMDB0000002,Pubchem,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
863754,HMDB0259856,E7FB98,ASSOCIATED_WITH,Metabolite,Protein,HMDB,HMDB0259856,Pubchem,NaN,NaN
863755,HMDB0259856,Q9Y5Z9,ASSOCIATED_WITH,Metabolite,Protein,HMDB,HMDB0259856,Pubchem,NaN,NaN
863756,HMDB0259928,P40765,ASSOCIATED_WITH,Metabolite,Protein,HMDB,HMDB0259928,Pubchem,NaN,NaN
863757,HMDB0259928,P53621,ASSOCIATED_WITH,Metabolite,Protein,HMDB,HMDB0259928,Pubchem,NaN,NaN


---
## File 33 — Protein → Protein (IntAct interactions)

In [43]:
df33 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/intact_interacts_with.tsv'),
    head_type='Protein', tail_type='Protein')

df33['Head_Alt_name'] = df33['Head'].map(Uniprot_Recname_dict)
df33['Tail_Alt_name'] = df33['Tail'].map(Uniprot_Recname_dict)
df33['KG_Source']  = 'CKG'
df33['Head_ID_IS'] = 'Uniprot_ID'
df33['Tail_ID_IS'] = 'Uniprot_ID'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_Alt_name","Head_ID_IS","Tail_ID_IS","evidence","response","disease"]
df33 = select_cols(df33, desired)
save(df33, 'File_33_Protein_Protein_CKG.csv')
df33

Saved 670,002 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_33_Protein_Protein_CKG.csv


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_Alt_name,Tail_Alt_name,Head_ID_IS,Tail_ID_IS
0,P13498,CURATED_INTERACTS_WITH,Q8NFA2,Protein,Protein,MINT,CKG,Cytochrome b-245 light chain,NADPH oxidase organizer 1,Uniprot_ID,Uniprot_ID
1,Q8TAU3,CURATED_INTERACTS_WITH,Q5JXC2,Protein,Protein,IntAct,CKG,Zinc finger protein 417,Migration and invasion-inhibitory protein,Uniprot_ID,Uniprot_ID
2,Q9NWW5,CURATED_INTERACTS_WITH,P39656,Protein,Protein,IntAct,CKG,Ceroid-lipofuscinosis neuronal protein 6,Dolichyl-diphosphooligosaccharide--protein gly...,Uniprot_ID,Uniprot_ID
3,P14678,CURATED_INTERACTS_WITH,Q8WUD4,Protein,Protein,UniProt,CKG,Small nuclear ribonucleoprotein-associated pro...,Coiled-coil domain-containing protein 12,Uniprot_ID,Uniprot_ID
4,P67870,CURATED_INTERACTS_WITH,Q8WXI9,Protein,Protein,UniProt,CKG,Casein kinase II subunit beta,Transcriptional repressor p66-beta,Uniprot_ID,Uniprot_ID
...,...,...,...,...,...,...,...,...,...,...,...
669997,P17948,CURATED_INTERACTS_WITH,Q13610,Protein,Protein,IntAct,CKG,Vascular endothelial growth factor receptor 1,Periodic tryptophan protein 1 homolog {ECO:000...,Uniprot_ID,Uniprot_ID
669998,O60895,CURATED_INTERACTS_WITH,Q12772,Protein,Protein,IntAct,CKG,Receptor activity-modifying protein 2,Processed sterol regulatory element-binding pr...,Uniprot_ID,Uniprot_ID
669999,Q96EK5,CURATED_INTERACTS_WITH,O43896,Protein,Protein,IntAct,CKG,KIF-binding protein,Kinesin-like protein KIF1C,Uniprot_ID,Uniprot_ID
670000,P41218,CURATED_INTERACTS_WITH,Q96GQ7,Protein,Protein,IntAct,CKG,Myeloid cell nuclear differentiation antigen,Probable ATP-dependent RNA helicase DDX27,Uniprot_ID,Uniprot_ID


In [91]:
# del df33

---
## File 34 — Mutation → Protein (curated, affects interaction)

In [121]:
df34 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/mutation_curated_affects_interaction_with.tsv'),
    head_type='Mutation', tail_type='Protein')

df34['Tail_Alt_name'] = df34['Tail'].map(Uniprot_Recname_dict)
df34['KG_Source']  = 'CKG'
df34['Head_ID_IS'] = 'Mutation'
df34['Tail_ID_IS'] = 'Uniprot_ID'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_Alt_name","Head_ID_IS","Tail_ID_IS",
           "effect","internal_id","evidence","response","disease","interaction"]
df34 = select_cols(df34, desired)
save(df34, 'File_34_Mutation_Protein_CKG.csv')
df34

Saved 135,598 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_34_Mutation_Protein_CKG.csv


,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Tail_Alt_name,Head_ID_IS,Tail_ID_IS,effect,internal_id,evidence,interaction
0,P49736_p.[Asp80Ala;Tyr81Ala],CURATED_AFFECTS_INTERACTION_WITH,Q9NVP2,Mutation,Protein,Intact-MutationDs,CKG,Histone chaperone ASF1B {ECO:0000305},Mutation,Uniprot_ID,mutation disrupting strength(MI:1128),EBI-16164801,26167883,"uniprotkb:P49736(protein(MI:0326), 9606 - Homo..."
1,Q14457_p.Phe123Ala,CURATED_AFFECTS_INTERACTION_WITH,Q16611,Mutation,Protein,Intact-MutationDs,CKG,Bcl-2 homologous antagonist/killer,Mutation,Uniprot_ID,mutation disrupting(MI:0573),EBI-5234926,17446862,"uniprotkb:Q07817-1(protein(MI:0326), 9606 - Ho..."
2,Q9NZ01_p.Pro182Leu,CURATED_AFFECTS_INTERACTION_WITH,Q9NZ01,Mutation,Protein,Intact-MutationDs,CKG,Very-long-chain enoyl-CoA reductase {ECO:0000305},Mutation,Uniprot_ID,mutation disrupting strength(MI:1128),EBI-24812746,32296183,"uniprotkb:Q9NZ01(protein(MI:0326), 9606 - Homo..."
3,O60656-1_p.Met33Thr,CURATED_AFFECTS_INTERACTION_WITH,P16662,Mutation,Protein,Intact-MutationDs,CKG,UDP-glucuronosyltransferase 2B7 {ECO:0000303|P...,Mutation,Uniprot_ID,mutation with no effect(MI:2226),EBI-48418883,27857056,"uniprotkb:O60656-1(protein(MI:0326), 9606 - Ho..."
4,Q5S007_p.Gly2019Ser,CURATED_AFFECTS_INTERACTION_WITH,Q59GY2,Mutation,Protein,Intact-MutationDs,CKG,NaN,Mutation,Uniprot_ID,mutation(MI:0118),EBI-21368474,31046837,"uniprotkb:A8K8U1(protein(MI:0326), 9606 - Homo..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135593,Q9H8Y8_p.Met1delinsGlyMet,CURATED_AFFECTS_INTERACTION_WITH,Q9H8Y8,Mutation,Protein,Intact-MutationDs,CKG,Golgi reassembly-stacking protein 2,Mutation,Uniprot_ID,mutation(MI:0118),EBI-70792362,36115835,"uniprotkb:Q9Y6M7(protein(MI:0326), 9606 - Homo..."
135594,P61019_p.Gln65Leu,CURATED_AFFECTS_INTERACTION_WITH,Q8R2X8,Mutation,Protein,Intact-MutationDs,CKG,Golgin-45,Mutation,Uniprot_ID,mutation increasing(MI:0382),EBI-4423356,11739401,"uniprotkb:P61019(protein(MI:0326), 9606 - Homo..."
135595,P51532_p.Arg885Cys,CURATED_AFFECTS_INTERACTION_WITH,Q13620,Mutation,Protein,Intact-MutationDs,CKG,Cullin-4B {ECO:0000305},Mutation,Uniprot_ID,mutation with no effect(MI:2226),EBI-63724217,35512704,"uniprotkb:P51532(protein(MI:0326), 9606 - Homo..."
135596,P13569_p.Phe508del,CURATED_AFFECTS_INTERACTION_WITH,P09874,Mutation,Protein,Intact-MutationDs,CKG,"Poly [ADP-ribose] polymerase 1, processed N-te...",Mutation,Uniprot_ID,mutation(MI:0118),EBI-27087557,31324722,"uniprotkb:Q9Y5S2(protein(MI:0326), 9606 - Homo..."


In [122]:
del df34

---
## File 35 — RefSeq located_in Chromosome → **DROPPED**

Chromosome node type is not in EvoAge KG scope.

---
## File 36 — Gene → Transcript (RefSeq transcribed into)

In [47]:

# df36 = process_dataframe(
#     os.path.join(BASE_PATH, 'ckg/docker_pulled/imports/databases/refseq_transcribed_into.tsv'),
#     head_type='Gene', tail_type='Ref_Sequence')

# df36 = resolve_gene_head(df36)
# df36['KG_Source']  = 'CKG'
# df36['Tail_ID_IS'] = 'NCBI Transcript ID'

# desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
#            "Head_Alt_name","Head_IUPAC_name","Head_Detail_name","Tail_Alt_name","Tail_Alt_ids",
#            "Head_ID_IS","Tail_ID_IS","effect","internal_id","evidence","response","disease","interaction"]
# df36 = select_cols(df36, desired)
# save(df36, 'File_36_Gene_Transcript_CKG.csv')
# del df36

---
## File 37 — Drug → Phenotype / Side Effect (SIDER)

In [48]:
df37 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/sider_is_indicated_for.tsv'),
    head_type='Drug', tail_type='Phenotype')

df37['Tail_Alt_name'] = df37['Tail'].map(phenotype_dict)
df37 = resolve_drug_head(df37)
df37.rename(columns={'Head_pubchem_mapped': "Head_Alt_ID's"}, inplace=True, errors='ignore')
df37['KG_Source']  = 'CKG'
df37['Tail_ID_IS'] = 'HPO_ID'

# BUG FIX: original listed "Head_Alt_ID's" TWICE in desired_cols
desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_ID's","Head_IUPAC","Head_Smiles","original_side_effect","Tail_Alt_name",
           "Tail_Detail_name","Head_ID_IS","Tail_ID_IS","evidence","response","disease"]
df37 = select_cols(df37, desired)
# save(df37, 'File_37_Drug_Phenotype_sideffect_CKG.csv')
# del df37
df37

,Head,Relation,Tail,Head_type,Tail_type,Source,KG_Source,Head_Alt_ID's,Head_IUPAC,Head_Smiles,original_side_effect,Tail_Alt_name,Head_ID_IS,Tail_ID_IS,evidence
0,5284373,IS_INDICATED_FOR,HP:0012123,Drug,Phenotype,SIDER,CKG,DB00091,CC[C@H]1C(=O)N(CC(=O)N([C@H](C(=O)N[C@H](C(=O)...,"(3S,6S,9S,12R,15S,18S,21S,24S,30S,33S)-30-ethy...",C0042167,Posterior uveitis,Pubchem,HPO_ID,NLP_indication
1,31101,IS_INDICATED_FOR,HP:0000798,Drug,Phenotype,SIDER,CKG,DB01200,CC(C)C[C@H]1C(=O)N2CCC[C@H]2[C@]3(N1C(=O)[C@](...,"(6aR,9R)-5-bromo-N-[(1S,2S,4R,7S)-2-hydroxy-7-...",C0028960,Oligospermia,Pubchem,HPO_ID,text_mention
2,9782,IS_INDICATED_FOR,HP:0011024,Drug,Phenotype,SIDER,CKG,DB00443,C[C@H]1C[C@H]2[C@@H]3CCC4=CC(=O)C=C[C@@]4([C@]...,"(8S,9R,10S,11S,13S,14S,16S,17R)-9-fluoro-11,17...",C0017178,Abnormality of the gastrointestinal tract,Pubchem,HPO_ID,text_mention
3,5288826,IS_INDICATED_FOR,HP:0002098,Drug,Phenotype,SIDER,CKG,DB00295,CN1CC[C@]23[C@@H]4[C@H]1CC5=C2C(=C(C=C5)O)O[C@...,"(4R,4aR,7S,7aR,12bS)-3-methyl-2,4,4a,7,7a,13-h...",C0013404,Respiratory distress,Pubchem,HPO_ID,NLP_indication
4,447043,IS_INDICATED_FOR,HP:0004469,Drug,Phenotype,SIDER,CKG,DB00207,CC[C@@H]1[C@@]([C@@H]([C@H](N(C[C@@H](C[C@@]([...,"(2R,3S,4R,5R,8R,10R,11R,12S,13S,14R)-11-[(2S,3...",C0008677,Chronic bronchitis,Pubchem,HPO_ID,text_mention
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6563,5253,IS_INDICATED_FOR,HP:0000083,Drug,Phenotype,SIDER,CKG,DB00489,CC(C)NCC(C1=CC=C(C=C1)NS(=O)(=O)C)O,N-[4-[1-hydroxy-2-(propan-2-ylamino)ethyl]phen...,C1565489,Renal insufficiency,Pubchem,HPO_ID,text_mention
6564,9782,IS_INDICATED_FOR,HP:0002488,Drug,Phenotype,SIDER,CKG,DB00443,C[C@H]1C[C@H]2[C@@H]3CCC4=CC(=O)C=C[C@@]4([C@]...,"(8S,9R,10S,11S,13S,14S,16S,17R)-9-fluoro-11,17...",C0085669,Acute leukemia,Pubchem,HPO_ID,text_mention
6565,2244,IS_INDICATED_FOR,HP:0001297,Drug,Phenotype,SIDER,CKG,DB00945,CC(=O)OC1=CC=CC=C1C(=O)O,2-acetyloxybenzoic acid,C0038454,Stroke,Pubchem,HPO_ID,text_mention
6566,3117,IS_INDICATED_FOR,HP:0002013,Drug,Phenotype,SIDER,CKG,DB00822,CCN(CC)C(=S)SSC(=S)N(CC)CC,"diethylcarbamothioylsulfanyl N,N-diethylcarbam...",C0042963,Vomiting,Pubchem,HPO_ID,text_mention


In [49]:
save(df37, 'File_37_Drug_Phenotype_sideffect_CKG.csv')
del df37

Saved 6,568 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_37_Drug_Phenotype_sideffect_CKG.csv


---
## Files 38 & 39 — Drug → Protein (STITCH associated / acts on)

In [50]:
def process_drug_protein_stitch(file_path, out_filename):
    """Shared pipeline for Drug->Protein STITCH files (38 and 39)."""
    df = process_dataframe(file_path, head_type='Drug', tail_type='Protein')
    df = resolve_drug_head(df)
    df['Tail_Alt_name'] = df['Tail'].map(Uniprot_Recname_dict)
    df.rename(columns={
        'interaction_type': 'Relation_type',
        'Head_pubchem_mapped': "Head_Alt_IDs"
    }, inplace=True, errors='ignore')
    df['KG_Source']  = 'CKG'
    df['Tail_ID_IS'] = 'UniprotID'
    # BUG FIX: original listed "Head_Alt_ID's" TWICE — deduplicated below
    desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
               "Head_Alt_IDs","Head_IUPAC","Head_Smiles","Tail_Alt_name","Head_ID_IS","Tail_ID_IS",
               "evidence","response","disease"]
    df = select_cols(df, desired)
    save(df, out_filename)

process_drug_protein_stitch(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/stitch_associated_with.tsv'),
    'File_38_Drug_Protein_CKG.csv')

process_drug_protein_stitch(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/stitch_drug_acts_on_protein.tsv'),
    'File_39_Drug_Protein_CKG.csv')

Saved 832,817 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_38_Drug_Protein_CKG.csv
Saved 1,659,588 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_39_Drug_Protein_CKG.csv


---
## Files 40 & 41 — Protein → Protein (STRING interactions / acts on)

In [51]:
def process_protein_protein_string(file_path, out_filename):
    """Shared pipeline for Protein->Protein STRING files (40 and 41)."""
    df = process_dataframe(file_path, head_type='Protein', tail_type='Protein')
    df['Head_Alt_name'] = df['Head'].map(Uniprot_Recname_dict)
    df['Tail_Alt_name'] = df['Tail'].map(Uniprot_Recname_dict)
    df.rename(columns={'action': 'Relation_type'}, inplace=True, errors='ignore')
    df['KG_Source']  = 'CKG'
    df['Head_ID_IS'] = 'UniprotID'
    df['Tail_ID_IS'] = 'UniprotID'
    desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
               "Head_Alt_IDs","Head_Alt_name","Head_IUPAC_name","Head_Detail_name",
               "Tail_Alt_name","Tail_Alt_ids","Head_ID_IS","Tail_ID_IS",
               "effect","internal_id","evidence","score","response","disease","interaction"]
    df = select_cols(df, desired)
    save(df, out_filename)

process_protein_protein_string(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/string_interacts_with.tsv'),
    'File_40_Protein_Protein_CKG.csv')

process_protein_protein_string(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/string_protein_acts_on_protein.tsv'),
    'File_41_Protein_Protein_CKG.csv')

Saved 52,090,442 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_40_Protein_Protein_CKG.csv
Saved 25,182,806 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_41_Protein_Protein_CKG.csv


---
## File 42 — PMID → Tissue (publication mention)

In [52]:
# instead of processed_df42. Fixed to use df42 throughout.
df42 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Tissue_Publication_mentioned_in_publication.tsv'),
    head_type='PMID', tail_type='Tissue')

# Keep only BTO-prefixed tissue IDs
df42 = df42[df42['Tail'].str.startswith('BTO')]
df42['Head'] = 'PMID:' + df42['Head'].astype(str)
df42['Tail_detail_name'] = df42['Tail'].map(BTO_dict)
df42 = df42[~df42['Tail_detail_name'].isna()]
df42['KG_Source']  = 'CKG'
df42['Head_ID_IS'] = 'PubmedID'
df42['Tail_ID_IS'] = 'BrendaTO'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Tail_detail_name","Head_ID_IS","Tail_ID_IS"]
df42 = select_cols(df42, desired)
save(df42, 'File_42_PMID_Tissue_CKG.csv')
df42

Saved 303,745,858 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_42_PMID_Tissue_CKG.csv


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,PMID:20174579,MENTIONED_IN_PUBLICATION,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO
1,PMID:20100909,MENTIONED_IN_PUBLICATION,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO
2,PMID:19597572,MENTIONED_IN_PUBLICATION,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO
3,PMID:19597582,MENTIONED_IN_PUBLICATION,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO
4,PMID:19357781,MENTIONED_IN_PUBLICATION,BTO:0000003,PMID,Tissue,CKG,intestinal cell line,PubmedID,BrendaTO
...,...,...,...,...,...,...,...,...,...
303745853,PMID:18566133,MENTIONED_IN_PUBLICATION,BTO:0006321,PMID,Tissue,CKG,GC-1 spg cell,PubmedID,BrendaTO
303745854,PMID:16395525,MENTIONED_IN_PUBLICATION,BTO:0006321,PMID,Tissue,CKG,GC-1 spg cell,PubmedID,BrendaTO
303745855,PMID:15991248,MENTIONED_IN_PUBLICATION,BTO:0006321,PMID,Tissue,CKG,GC-1 spg cell,PubmedID,BrendaTO
303745856,PMID:11179488,MENTIONED_IN_PUBLICATION,BTO:0006321,PMID,Tissue,CKG,GC-1 spg cell,PubmedID,BrendaTO


In [53]:
del df42

# PMID DRUG

In [4]:
df43 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Drug_Publication_mentioned_in_publication.tsv'),
    head_type='PMID', tail_type='Drug')
df43
df43['Head'] = 'PMID:' + df43['Head'].astype(str)

,Head,Tail,Relation,Head_type,Tail_type
0,20398344,DB08842,MENTIONED_IN_PUBLICATION,PMID,Drug
1,19399243,DB08842,MENTIONED_IN_PUBLICATION,PMID,Drug
2,19783937,DB08842,MENTIONED_IN_PUBLICATION,PMID,Drug
3,18555826,DB08842,MENTIONED_IN_PUBLICATION,PMID,Drug
4,19300586,DB08842,MENTIONED_IN_PUBLICATION,PMID,Drug
...,...,...,...,...,...
48560314,37652915,DB08823,MENTIONED_IN_PUBLICATION,PMID,Drug
48560315,41260918,DB08823,MENTIONED_IN_PUBLICATION,PMID,Drug
48560316,21980712,DB08823,MENTIONED_IN_PUBLICATION,PMID,Drug
48560317,21701441,DB08823,MENTIONED_IN_PUBLICATION,PMID,Drug


In [16]:
df43 = resolve_drug_tail(df43)
df43
# df1 = resolve_gene_tail(df1)
df43['KG_Source'] = 'CKG'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alias_mapped","Head_Alt_name","Head_Detail_name","Head_IUPAC","Head_Smiles","Head_ENS",
           "Tail_Detail_name","Tail_Alias_mapped","Tail_DO_Alt_ids","Tail_Alt_name","Head_ID_IS","Tail_ID_IS",
           "evidence_type","score","diseaseType"]

NameError: name 'df1' is not defined

In [17]:
df43 = select_cols(df43, desired)
df43

,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Tail_ID_IS
0,20398344,MENTIONED_IN_PUBLICATION,7045767,PMID,Drug,CKG,Pubchem
1,19399243,MENTIONED_IN_PUBLICATION,7045767,PMID,Drug,CKG,Pubchem
2,19783937,MENTIONED_IN_PUBLICATION,7045767,PMID,Drug,CKG,Pubchem
3,18555826,MENTIONED_IN_PUBLICATION,7045767,PMID,Drug,CKG,Pubchem
4,19300586,MENTIONED_IN_PUBLICATION,7045767,PMID,Drug,CKG,Pubchem
...,...,...,...,...,...,...,...
48560314,37652915,MENTIONED_IN_PUBLICATION,17754356,PMID,Drug,CKG,Pubchem
48560315,41260918,MENTIONED_IN_PUBLICATION,17754356,PMID,Drug,CKG,Pubchem
48560316,21980712,MENTIONED_IN_PUBLICATION,17754356,PMID,Drug,CKG,Pubchem
48560317,21701441,MENTIONED_IN_PUBLICATION,17754356,PMID,Drug,CKG,Pubchem


In [18]:
df43['Head'] = 'PMID:' + df43['Head'].astype(str)
df43['Tail_detail_name'] = df43['Tail'].map(Pubchem_IUPAC_CID_Dict)
df43 = df43[~df43['Tail_detail_name'].isna()]
df43['KG_Source']  = 'CKG'
df43['Head_ID_IS'] = 'PubmedID'
df43['Tail_ID_IS'] = 'Pubchem'
df43

/tmp/ipykernel_3066890/3822345789.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df43['KG_Source']  = 'CKG'
/tmp/ipykernel_3066890/3822345789.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df43['Head_ID_IS'] = 'PubmedID'
/tmp/ipykernel_3066890/3822345789.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guid

,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Tail_ID_IS,Tail_detail_name,Head_ID_IS
0,PMID:20398344,MENTIONED_IN_PUBLICATION,7045767,PMID,Drug,CKG,Pubchem,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,PubmedID
1,PMID:19399243,MENTIONED_IN_PUBLICATION,7045767,PMID,Drug,CKG,Pubchem,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,PubmedID
2,PMID:19783937,MENTIONED_IN_PUBLICATION,7045767,PMID,Drug,CKG,Pubchem,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,PubmedID
3,PMID:18555826,MENTIONED_IN_PUBLICATION,7045767,PMID,Drug,CKG,Pubchem,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,PubmedID
4,PMID:19300586,MENTIONED_IN_PUBLICATION,7045767,PMID,Drug,CKG,Pubchem,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,PubmedID
...,...,...,...,...,...,...,...,...,...
48560314,PMID:37652915,MENTIONED_IN_PUBLICATION,17754356,PMID,Drug,CKG,Pubchem,"(1S,2S,5R,7S,9S,10S,14R,15S,19S)-15-[(2R,5S,6R...",PubmedID
48560315,PMID:41260918,MENTIONED_IN_PUBLICATION,17754356,PMID,Drug,CKG,Pubchem,"(1S,2S,5R,7S,9S,10S,14R,15S,19S)-15-[(2R,5S,6R...",PubmedID
48560316,PMID:21980712,MENTIONED_IN_PUBLICATION,17754356,PMID,Drug,CKG,Pubchem,"(1S,2S,5R,7S,9S,10S,14R,15S,19S)-15-[(2R,5S,6R...",PubmedID
48560317,PMID:21701441,MENTIONED_IN_PUBLICATION,17754356,PMID,Drug,CKG,Pubchem,"(1S,2S,5R,7S,9S,10S,14R,15S,19S)-15-[(2R,5S,6R...",PubmedID


In [19]:
save(df43, 'File_43_PMID_CHEMICAL_CKG.csv')

Saved 42,809,926 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_43_PMID_CHEMICAL_CKG.csv


In [28]:
del df43

# PMID Protein

In [22]:
# instead of processed_df42. Fixed to use df42 throughout.
df44 = process_dataframe(
    os.path.join(BASE_PATH, 'ckg/data/imports/databases/Protein_Publication_mentioned_in_publication.tsv'),
    head_type='PMID', tail_type='Protein')
df44
df44['Head'] = 'PMID:' + df44['Head'].astype(str)

,Head,Tail,Relation,Head_type,Tail_type
0,20463971,P84085,MENTIONED_IN_PUBLICATION,PMID,Protein
1,17555535,P84085,MENTIONED_IN_PUBLICATION,PMID,Protein
2,15644143,P84085,MENTIONED_IN_PUBLICATION,PMID,Protein
3,20214810,P84085,MENTIONED_IN_PUBLICATION,PMID,Protein
4,20089835,P84085,MENTIONED_IN_PUBLICATION,PMID,Protein
...,...,...,...,...,...
237292205,8536891,Q8N4M9,MENTIONED_IN_PUBLICATION,PMID,Protein
237292206,6429045,Q8N4M9,MENTIONED_IN_PUBLICATION,PMID,Protein
237292207,31591430,A0A096LPK9,MENTIONED_IN_PUBLICATION,PMID,Protein
237292208,38259516,A0A096LPK9,MENTIONED_IN_PUBLICATION,PMID,Protein


In [26]:
df44['Tail_Alt_name']   = df44['Tail'].map(Uniprot_Recname_dict)

df44['KG_Source']  = 'CKG'
df44['Head_ID_IS'] = 'PMID'
df44['Tail_ID_IS'] = 'Uniprot_ID'

desired = ["Head","Relation","Tail","Head_type","Relation_type","Tail_type","Source","KG_Source",
           "Head_Alt_name","Tail_DO_name","Tail_Alt_name","Head_ID_IS","Tail_ID_IS","evidence_type","score"]
df44 = select_cols(df44, desired)
df44

,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Tail_Alt_name,Head_ID_IS,Tail_ID_IS
0,20463971,MENTIONED_IN_PUBLICATION,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID
1,17555535,MENTIONED_IN_PUBLICATION,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID
2,15644143,MENTIONED_IN_PUBLICATION,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID
3,20214810,MENTIONED_IN_PUBLICATION,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID
4,20089835,MENTIONED_IN_PUBLICATION,P84085,PMID,Protein,CKG,ADP-ribosylation factor 5,PMID,Uniprot_ID
...,...,...,...,...,...,...,...,...,...
237292205,8536891,MENTIONED_IN_PUBLICATION,Q8N4M9,PMID,Protein,CKG,Mucin-5AC {ECO:0000305},PMID,Uniprot_ID
237292206,6429045,MENTIONED_IN_PUBLICATION,Q8N4M9,PMID,Protein,CKG,Mucin-5AC {ECO:0000305},PMID,Uniprot_ID
237292207,31591430,MENTIONED_IN_PUBLICATION,A0A096LPK9,PMID,Protein,CKG,Olfactory receptor 4N4C {ECO:0000305},PMID,Uniprot_ID
237292208,38259516,MENTIONED_IN_PUBLICATION,A0A096LPK9,PMID,Protein,CKG,Olfactory receptor 4N4C {ECO:0000305},PMID,Uniprot_ID


In [27]:
save(df44, 'File_44_PMID_PROTEIN_CKG.csv')

Saved 237,292,210 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/CKG/File_44_PMID_PROTEIN_CKG.csv


---
## 3 · Summary — all output files

In [123]:
import os, pandas as pd
from glob import glob

data_dir = OUT_PATH
cols = ["Relation","Head_type","Tail_type","Source","KG_Source","Head_ID_IS","Tail_ID_IS"]

df_list = []
for file_path in sorted(glob(os.path.join(data_dir, '*.csv'))):
    try:
        total_df   = pd.read_csv(file_path)
        total_rows = len(total_df)
        temp_df    = total_df.head(5)
        available  = [c for c in cols if c in temp_df.columns]
        row = {'Filename': os.path.basename(file_path), 'no. of triples': total_rows}
        for c in available:
            row[c] = set(temp_df[c].dropna().tolist())
        df_list.append(pd.DataFrame([row]))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

combined_df = pd.concat(df_list, ignore_index=True)
print(f"Total files: {len(combined_df)}")
combined_df

/tmp/ipykernel_1970976/2646669414.py:10: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  total_df   = pd.read_csv(file_path)

KeyboardInterrupt

